In [ ]:
# ============================================================
# CELL 0 — PATH CONFIGURATION (edit these to match your own setup)
# Run this FIRST. Every other cell references these variables, so
# nothing else needs to be touched if you change a path here.
# ============================================================

import os
from pathlib import Path
from google.colab import drive

# ---- Hugging Face dataset (raw IR/Depth training data) ----
HF_DATASET_REPO = "Kevin-Pal/CUHK-X_Small_Model_Track"   # dataset repo on huggingface.co
DOWNLOAD_TEST_DATA = False   # this experiment only needs Training/ -- set True to also pull Testing/

# ---- local Colab paths for the raw data (staging + extraction) ----
LOCAL_TRAINING = Path("/content/Training")
LOCAL_DATA = LOCAL_TRAINING / "data"
EXTRACT_ROOT = Path("/content/train_extracted")

# ---- Google Drive: still used as the working cache for checkpoints,
# predictions and oracle results across Colab sessions -- unrelated to the
# raw-data source above, which now comes from Hugging Face instead of Drive.
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/CUHKX_cache/gate_system_per_model"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
PRED_DIR = f"{DRIVE_ROOT}/predictions"
ORACLE_DIR = f"{DRIVE_ROOT}/oracle"
PROGRESS_PATH = f"{DRIVE_ROOT}/progress.json"
for d in [CKPT_DIR, PRED_DIR, ORACLE_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"HF_DATASET_REPO = {HF_DATASET_REPO}")
print(f"EXTRACT_ROOT    = {EXTRACT_ROOT}")
print(f"DRIVE_ROOT      = {DRIVE_ROOT}")
print("✓ Path config ready.")


In [ ]:
# ============================================================
# CELL 1 — DOWNLOAD RAW DATA FROM HUGGING FACE + EXTRACT
# Replaces the old Google-Drive-based raw-data cell. The Hugging Face
# dataset (see CELL 0 for the repo id) is an exact mirror of what used
# to sit on Drive, so everything downstream (Cell 2 onward) is unchanged.
#
# Repo layout:
#   Training/class_mapping.csv                      (action_id <-> action_name)
#   Training/data/HAR.z01 ... HAR.z08 + HAR.zip      (multi-volume zip)
#     -> extracts to HAR/data/<modality>/<action>/<user>/<trial>/<files>
#   Testing/... (only pulled if DOWNLOAD_TEST_DATA = True in Cell 0)
# ============================================================

import subprocess, shutil
from pathlib import Path

try:
    from huggingface_hub import snapshot_download
except ImportError:
    subprocess.run(["pip", "install", "-q", "-U", "huggingface_hub"])
    from huggingface_hub import snapshot_download

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("1.1 - HUGGING FACE DATASET DOWNLOAD")
patterns = ["Training/data/*", "Training/class_mapping.csv"]
if DOWNLOAD_TEST_DATA:
    patterns += ["Testing/**"]

# If the dataset repo is gated/private, log in first with:
#   from huggingface_hub import login; login()
hf_local_root = Path(
    snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset", allow_patterns=patterns)
)
print(f"✓ Downloaded to HF cache: {hf_local_root}", flush=True)

stage("1.2 - STAGING FILES INTO THE EXPECTED LOCAL LAYOUT")
already_extracted = EXTRACT_ROOT.exists() and any(EXTRACT_ROOT.iterdir())
already_copied = LOCAL_DATA.exists() and any(LOCAL_DATA.glob("*.zip"))

if already_extracted:
    print(f"✓ {EXTRACT_ROOT} already extracted, skipping staging.", flush=True)
else:
    for path in [LOCAL_TRAINING, EXTRACT_ROOT]:
        if path.exists():
            print(f"Removing: {path}", flush=True)
            shutil.rmtree(path)
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    src_data = hf_local_root / "Training" / "data"
    for f in src_data.iterdir():
        shutil.copy2(f, LOCAL_DATA / f.name)
    print(f"✓ Staged {len(list(LOCAL_DATA.iterdir()))} files into {LOCAL_DATA}", flush=True)

stage("1.3 - CHECKING ARCHIVES")
zip_files = sorted(LOCAL_DATA.glob("*.zip"))
split_parts = sorted(LOCAL_DATA.glob("*.z[0-9][0-9]"))
assert len(zip_files) > 0, f"No .zip found in {LOCAL_DATA}."
print(f"✓ Main ZIPs: {len(zip_files)}  Split parts: {len(split_parts)}", flush=True)
for p in split_parts + zip_files:
    assert p.stat().st_size > 0, f"Empty file: {p}"
    print(f"  {p.name:<15} {p.stat().st_size / 1024**3:.2f} GiB", flush=True)

stage("1.4 - PREPARING 7-ZIP")
if shutil.which("7z") is None:
    result = subprocess.run(["apt-get", "install", "-y", "-qq", "p7zip-full"])
    assert result.returncode == 0
print("✓ 7-Zip ready.", flush=True)

stage("1.5 - EXTRACTING TRAINING DATA")
# HAR.zip is the "head" volume; HAR.z01..z08 sit next to it in the same
# folder and 7z follows the chain automatically as long as all parts are
# present together -- this is unchanged from the old Drive-based cell.
if already_extracted:
    print("✓ Data already extracted, skipping 7z.", flush=True)
else:
    for i, zip_path in enumerate(zip_files, start=1):
        print(f"\n[{i}/{len(zip_files)}] Extracting: {zip_path.name}", flush=True)
        result = subprocess.run(["7z", "x", str(zip_path), f"-o{EXTRACT_ROOT}", "-aoa", "-bsp1", "-bb1"])
        assert result.returncode == 0, f"Extraction failed: {zip_path}"
        print(f"✓ {zip_path.name} done.", flush=True)

extracted_files = sum(p.is_file() for p in EXTRACT_ROOT.rglob("*"))
assert extracted_files > 0, "Extraction reported success but no files exist."
stage("DATASET READY")
print(f"✓ Extracted file count: {extracted_files:,}", flush=True)
print(f"✓ Extracted data at: {EXTRACT_ROOT}", flush=True)


In [ ]:
# ============================================================
# CELL 2/20 — DATA SCAN + YOLO CROP CACHE (IR and Depth, SEPARATELY)
# Run after the data-download cell. Computes from scratch, no old cache is read.
#
# IMPORTANT: in this dataset each clip is NOT a single video file, it is a
# folder containing MANY frame (PNG) files:
#   Depth_Color/0_Wash_face/user16/1-1-1/Depth_2025-06-10_10-43-49.016_00000151_Color.png
#   Depth_Color/0_Wash_face/user16/1-1-1/Depth_2025-06-10_10-43-49.116_00000152_Color.png
#   ... (dozens of frames in the same "1-1-1" folder)
# So there is no video reading below -- ALL frames in a folder are collected
# and SORTED by the numeric frame-index in the filename (that index defines
# temporal order; the filename/timestamp's alphabetical order is NOT
# trustworthy).
# ============================================================

import re, time
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

# ------------------------------------------------------------------
# 2.1 — DATA SCAN: items / train_ids / val_ids / subject_to_id
# ------------------------------------------------------------------

stage("2.1 - VERİ TARAMA")

N_CLASSES = 40
VAL_USER_NUMBERS = {1, 8, 9, 21}
CACHE_IMAGE_SIZE = 160     # written to the RAM cache at this size; each
                           # architecture resizes to its own native
                           # resolution (112 / 224) during training.
CACHE_FRAMES = 24
TRAIN_FRAMES = 16
CROP_MARGIN = 1.50
IR_DIR_CANDIDATES = ["IR", "Infrared", "ir"]
DEPTH_DIR_CANDIDATES = ["Depth_Color", "DepthColor", "depth_color", "Depth"]

def find_modality_dir(root, candidates):
    for cand in candidates:
        hits = [p for p in root.rglob("*") if p.is_dir() and p.name.lower() == cand.lower()]
        if hits:
            hits.sort(key=lambda p: sum(1 for _ in p.rglob("*")), reverse=True)
            return hits[0]
    return None

ir_dir = find_modality_dir(EXTRACT_ROOT, IR_DIR_CANDIDATES)
depth_dir = find_modality_dir(EXTRACT_ROOT, DEPTH_DIR_CANDIDATES)
assert ir_dir is not None, f"IR klasörü bulunamadı. Aday: {IR_DIR_CANDIDATES}"
assert depth_dir is not None, f"Depth klasörü bulunamadı. Aday: {DEPTH_DIR_CANDIDATES}"
print(f"✓ IR klasörü    : {ir_dir}")
print(f"✓ Depth klasörü : {depth_dir}")

# -- matching is done by FOLDER HIERARCHY, NOT by filename --
# actual structure: Depth_Color/0_Wash_face/user16/1-1-1/Depth_..._00000151_Color.png
#                    IR/0_Wash_face/user16/1-1-1/IR_..._00000151(...).png
# i.e. the "action/user/trial" folder path is THE SAME between IR and Depth --
# since the two cameras may have recorded with different real timestamps and
# different frame counts, matching by filename would be WRONG, so matching is
# done by folder path instead. No extension is assumed either -- whatever is
# found is counted.

LABEL_FOLDER_PATTERN = re.compile(r"^(\d+)_")
USER_FOLDER_PATTERN = re.compile(r"user[_-]?(\d+)", re.IGNORECASE)
FRAME_INDEX_TOKEN_PATTERN = re.compile(r"^\d+$")

def parse_label_user_from_folders(folder_parts):
    """Looks only at FOLDER names (not the filename)."""
    label, user = None, None
    for part in folder_parts:
        if label is None:
            m = LABEL_FOLDER_PATTERN.match(part)
            if m: label = int(m.group(1))
        if user is None:
            m = USER_FOLDER_PATTERN.search(part)
            if m: user = int(m.group(1))
    return label, user

def extract_frame_index(path):
    """Takes the FULLY NUMERIC last token, split by '_', in the filename as
    the frame index -- e.g. '..._00000151_Color.png' -> 151. If there are
    multiple fully-numeric tokens, the LAST one is used (date/time parts
    usually contain '-' or '.' so they aren't purely numeric tokens and
    won't be confused with it). Returns None if none is found."""
    tokens = path.stem.split("_")
    frame_idx = None
    for tok in tokens:
        if FRAME_INDEX_TOKEN_PATTERN.match(tok):
            frame_idx = int(tok)
    return frame_idx

def sort_frames(paths):
    """Sorts frames by the NUMERIC frame-index in the filename (that is the
    temporal order). Falls back safely to alphabetical filename sort for any
    file with no index."""
    with_idx = [(extract_frame_index(p), p) for p in paths]
    if all(idx is not None for idx, _ in with_idx):
        return [p for _, p in sorted(with_idx, key=lambda t: t[0])]
    return sorted(paths, key=lambda p: p.name)

def scan_modality(modality_dir):
    """Finds ALL files under modality_dir (no extension assumed), groups each
    file by its parent folder path (action/user/trial), and sorts the frames
    within each group by frame-index.
    Returns: by_key {folder_path: [sorted frame files, ...]}, extension
    histogram, per-folder frame-count stats, unparsable entries."""
    raw_groups, ext_counter, label_miss, user_miss = {}, {}, [], []
    for p in modality_dir.rglob("*"):
        if not p.is_file():
            continue
        ext_counter[p.suffix.lower()] = ext_counter.get(p.suffix.lower(), 0) + 1
        folder_parts = p.parent.relative_to(modality_dir).parts   # folders ONLY, filename excluded
        label, user = parse_label_user_from_folders(folder_parts)
        if label is None:
            label_miss.append(str(p)); continue
        if user is None:
            user_miss.append(str(p)); continue
        key = str(p.parent.relative_to(modality_dir))
        raw_groups.setdefault(key, []).append(p)
    by_key = {key: sort_frames(paths) for key, paths in raw_groups.items()}
    return by_key, ext_counter, label_miss, user_miss

ir_by_key, ir_ext_counter, ir_label_miss, ir_user_miss = scan_modality(ir_dir)
depth_by_key, depth_ext_counter, depth_label_miss, depth_user_miss = scan_modality(depth_dir)

def frame_count_stats(by_key):
    counts = [len(v) for v in by_key.values()]
    if not counts: return "klasör yok"
    return f"min={min(counts)} max={max(counts)} ortalama={sum(counts)/len(counts):.1f}"

print(f"\n✓ IR    -- klasör(key) sayısı: {len(ir_by_key)}   uzantı dağılımı: {ir_ext_counter}   "
      f"klasör başına kare sayısı: {frame_count_stats(ir_by_key)}")
print(f"✓ Depth -- klasör(key) sayısı: {len(depth_by_key)}   uzantı dağılımı: {depth_ext_counter}   "
      f"klasör başına kare sayısı: {frame_count_stats(depth_by_key)}")
if ir_label_miss or ir_user_miss:
    print(f"  !! IR: label parse edilemeyen {len(ir_label_miss)}, user parse edilemeyen {len(ir_user_miss)} dosya. "
          f"Örnek: {(ir_label_miss + ir_user_miss)[:3]}")
if depth_label_miss or depth_user_miss:
    print(f"  !! Depth: label parse edilemeyen {len(depth_label_miss)}, user parse edilemeyen {len(depth_user_miss)} dosya. "
          f"Örnek: {(depth_label_miss + depth_user_miss)[:3]}")

assert len(ir_by_key) > 0, "IR altında hiçbir dosyadan (aksiyon/kullanıcı) parse edilemedi -- klasör isimlendirmesi beklenenden farklı, yukarıdaki örnekleri Claude'a gönder."
assert len(depth_by_key) > 0, "Depth altında hiçbir dosyadan (aksiyon/kullanıcı) parse edilemedi -- klasör isimlendirmesi beklenenden farklı, yukarıdaki örnekleri Claude'a gönder."

print("\n--- Örnek IR anahtarları (klasör yolu -> ilk 3 kare, sıralı) ---")
for k in list(ir_by_key.keys())[:5]: print(" ", k, f"({len(ir_by_key[k])} kare) ->", [p.name for p in ir_by_key[k][:3]])
print("--- Örnek Depth anahtarları (klasör yolu -> ilk 3 kare, sıralı) ---")
for k in list(depth_by_key.keys())[:5]: print(" ", k, f"({len(depth_by_key[k])} kare) ->", [p.name for p in depth_by_key[k][:3]])

common_keys = sorted(set(ir_by_key) & set(depth_by_key))
print(f"\n✓ Ortak klasör(key) sayısı: {len(common_keys)}  "
      f"(sadece IR: {len(set(ir_by_key)-set(depth_by_key))}, "
      f"sadece Depth: {len(set(depth_by_key)-set(ir_by_key))})")

if len(common_keys) == 0:
    print("IR key örnekleri   :", list(ir_by_key.keys())[:5])
    print("Depth key örnekleri:", list(depth_by_key.keys())[:5])
    raise AssertionError("IR/Depth klasör yolları eşleşmiyor -- klasör hiyerarşisi beklenenden farklı, yukarıdaki örnekleri Claude'a gönder.")

items, parse_failures = [], []
for key in common_keys:
    ir_frames = ir_by_key[key]
    depth_frames = depth_by_key[key]
    label, user = parse_label_user_from_folders(ir_frames[0].parent.relative_to(ir_dir).parts)
    if label is None or user is None or not (0 <= label < N_CLASSES):
        parse_failures.append(key); continue
    if len(ir_frames) == 0 or len(depth_frames) == 0:
        parse_failures.append(key); continue
    items.append({"key": key, "user": user, "label": label,
                   "ir_frames": [str(p) for p in ir_frames],
                   "depth_frames": [str(p) for p in depth_frames]})

print(f"\n✓ Parse edilen örnek: {len(items)}   Parse edilemeyen: {len(parse_failures)}")
print("\n--- İlk 10 örnek (GÖZLE KONTROL ET) ---")
for it in items[:10]:
    print(f"  key={it['key']:<40} user={it['user']:<4} label={it['label']:<3} "
          f"ir_kare={len(it['ir_frames']):<4} depth_kare={len(it['depth_frames'])}")
if parse_failures:
    print("\n--- Parse edilemeyen ilk 10 ---")
    for k in parse_failures[:10]: print(" ", k)
assert len(items) > 0, "Hiçbir örnek parse edilemedi -- klasör isimlendirme deseni uyuşmuyor."

unique_subjects = sorted(set(it["user"] for it in items))
print(f"\n✓ Toplam segment: {len(items)}  Subject: {len(unique_subjects)} -> {unique_subjects}")
print(f"✓ Label sayısı: {len(set(it['label'] for it in items))} -> {sorted(set(it['label'] for it in items))}")

train_ids = [i for i, it in enumerate(items) if it["user"] not in VAL_USER_NUMBERS]
val_ids = [i for i, it in enumerate(items) if it["user"] in VAL_USER_NUMBERS]
subject_to_id = {u: idx for idx, u in enumerate(unique_subjects)}
print(f"✓ train={len(train_ids)}  val={len(val_ids)}  subject_to_id={subject_to_id}")
assert len(train_ids) > 0 and len(val_ids) > 0, "train/val boş -- VAL_USER_NUMBERS kontrol et (mevcut subject listesiyle kesişmiyor olabilir)."


# ------------------------------------------------------------------
# 2.2 — YOLO CROP CACHE: run on IR ONLY.
# The clips are synchronized (aligned cameras filming the same scene at the
# same time) and YOLO person detection is more reliable on IR than on
# Depth_Color -- so the bbox found from IR is used for cropping BOTH IR and
# Depth (in Cell 3).
# The reference frame read is the MIDDLE frame in the folder (not a video --
# a single PNG, read with cv2.imread).
# Every clip is wrapped in try/except for ROBUST error handling (frame
# couldn't be read / YOLO crashed / unexpected error -- none of these should
# crash the whole loop).
# ------------------------------------------------------------------

stage("2.2 - YOLO CROP CACHE (sadece IR, sıfırdan)")

get_ipython().system("pip install -q ultralytics")
from ultralytics import YOLO

CACHE_DIR = Path("/content/drive/MyDrive/CUHKX_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ANCHOR_CSV_IR_PATH = CACHE_DIR / "anchor_boxes_ir.csv"
PERSON_CLASS_ID = 0

yolo_model = YOLO("yolov8x.pt")
anchor_lookup_ir = {}     # -- intentionally starts empty, no old cache is read --

N_REF_FRAMES_FOR_YOLO = 8   # -- a single frame is not trusted: YOLO is run on
                             # the first 8 frames and the UNION of the found
                             # bboxes is taken. This gives a tight crop based
                             # on the person's motion over these first 8
                             # frames -- reducing the risk of a bbox that's
                             # either too tight (cutting the person off, based
                             # on one random/bad frame) or unnecessarily wide
                             # (including too much background).

def compute_union_bbox(frame_paths, n_ref_frames, margin):
    """Runs YOLO person detection on the first n_ref_frames frames (read one
    by one with cv2.imread), and takes the union of all DETECTED bboxes
    (min x1/y1, max x2/y2). This produces a crop based on the ACTUAL area the
    person covers over these frames, not just a single frame. If no person is
    found in a given frame, that frame is simply excluded from the union (the
    others are enough); if NO frame finds a person, it falls back to the full
    frame. Every step is wrapped in try/except -- no single error can crash
    the whole loop."""
    ref_paths = frame_paths[:n_ref_frames]
    any_frame_shape, boxes, per_frame_errors = None, [], []
    for fp in ref_paths:
        try:
            frame = cv2.imread(str(fp))
        except Exception as e:
            per_frame_errors.append(f"{fp}: okuma_hatasi {e}"); continue
        if frame is None:
            per_frame_errors.append(f"{fp}: okunamadi"); continue
        if any_frame_shape is None:
            any_frame_shape = frame.shape[:2]
        try:
            results = yolo_model(frame, verbose=False)
        except Exception as e:
            per_frame_errors.append(f"{fp}: yolo_hatasi {e}"); continue
        best_box, best_conf = None, -1.0
        for r in results:
            if r.boxes is None:
                continue
            for box, cls, conf in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
                if int(cls) == PERSON_CLASS_ID and conf > best_conf:
                    best_conf, best_box = float(conf), box
        if best_box is not None:
            boxes.append(best_box)
        else:
            per_frame_errors.append(f"{fp}: kisi_bulunamadi")

    if not boxes:
        if any_frame_shape is not None:
            h, w = any_frame_shape
        else:
            h, w = CACHE_IMAGE_SIZE, CACHE_IMAGE_SIZE
        return (0, 0, w, h), "hicbir_karede_kisi_bulunamadi", per_frame_errors

    boxes = np.array(boxes)
    x1u, y1u = boxes[:, 0].min(), boxes[:, 1].min()
    x2u, y2u = boxes[:, 2].max(), boxes[:, 3].max()
    bw, bh = x2u - x1u, y2u - y1u
    cx, cy = x1u + bw / 2, y1u + bh / 2
    bw *= margin; bh *= margin
    h, w = any_frame_shape
    bbox = (max(0, cx - bw/2), max(0, cy - bh/2), min(w, cx + bw/2), min(h, cy + bh/2))
    err = None if len(boxes) == len(ref_paths) else f"{len(ref_paths)-len(boxes)}/{len(ref_paths)} karede kişi tespit edilemedi"
    return bbox, err, per_frame_errors

def cache_bboxes_for_modality(frames_key, anchor_lookup, csv_path, modality_label):
    """frames_key: 'ir_frames' or 'depth_frames' (the LIST of frame files).
    For every clip, catches every kind of error (frame couldn't be read / YOLO
    crashed / unexpected error) one at a time via try/except, falls back to a
    default bbox for that clip, and CONTINUES."""
    new_rows, error_log = [], []
    start = time.time()
    for i, it in enumerate(items, start=1):
        key = it["key"]
        frame_paths = it[frames_key]
        try:
            bbox, err, _per_frame_errors = compute_union_bbox(frame_paths, N_REF_FRAMES_FOR_YOLO, CROP_MARGIN)
            if err is not None:
                error_log.append((key, err))
        except Exception as e:
            # any UNEXPECTED error -- still continue with the fallback
            bbox = (0, 0, CACHE_IMAGE_SIZE, CACHE_IMAGE_SIZE)
            error_log.append((key, f"beklenmedik_hata: {e}"))

        anchor_lookup[key] = bbox
        new_rows.append({"key": key, "x1": bbox[0], "y1": bbox[1], "x2": bbox[2], "y2": bbox[3]})

        if i % 200 == 0 or i == len(items):
            elapsed = (time.time() - start) / 60
            print(f"  [{modality_label}] [{i}/{len(items)}] bbox hesaplandı "
                  f"({elapsed:.1f} dk, {len(error_log)} uyarı)")

    pd.DataFrame(new_rows).to_csv(csv_path, index=False)
    print(f"✓ [{modality_label}] {len(anchor_lookup)} klip cache'lendi -> {csv_path}")
    if error_log:
        print(f"  -- {len(error_log)} klipte tam/kısmi kişi tespiti sorunu (ilk 5):")
        for k, err in error_log[:5]:
            print(f"     {k}: {err}")
    return error_log

print("\n>>> IR için bbox hesaplanıyor (Depth de aynı bbox'ı kullanacak -- klipler senkron)...")
ir_yolo_errors = cache_bboxes_for_modality("ir_frames", anchor_lookup_ir, ANCHOR_CSV_IR_PATH, "IR")

def get_crop_box_ir(key, frame_shape=None):
    if key in anchor_lookup_ir:
        return anchor_lookup_ir[key]
    if frame_shape is not None:
        h, w = frame_shape[:2]; return (0, 0, w, h)
    return (0, 0, CACHE_IMAGE_SIZE, CACHE_IMAGE_SIZE)

# The Depth crop also uses IR's bbox -- since the cameras are synchronized and
# aligned, a separate YOLO pass isn't needed, giving a consistent crop from a
# single source.
get_crop_box_depth = get_crop_box_ir

print(f"\n✓ Hücre 2 tamamlandı: items, train_ids, val_ids, subject_to_id, "
      f"get_crop_box_ir() hazır (Depth de aynısını kullanıyor).")


In [ ]:
# ============================================================
# CELL 3/20 — RAM CACHE + SHARED DEFINITIONS FOR 6 ARCHITECTURES
# (Dataset class, model parts, training functions, Drive checkpointing,
#  oracle analysis -- all 12 cells after this one will use these)
# Run after the YOLO-crop-cache cell.
# ============================================================

import os, json, time, gc, copy, math
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models.video as video_models

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

# ------------------------------------------------------------------
# 3.1 — RAM CACHE (IR and Depth are each cropped with their own bbox)
# ------------------------------------------------------------------

stage("3.1 - RAM CACHE")

# IMPORTANT: a clip is NOT a single video file, it is a frame (PNG) LIST
# that Cell 2 has ALREADY SORTED by the numeric frame-index in the filename
# (it["ir_frames"] / it["depth_frames"]). There is no video reading here,
# frames are read one by one with cv2.imread -- the temporal order relies on
# the sort already established in Cell 2, no re-sorting happens.

def decode_clip_cropped(frame_paths, crop_box, n_frames, size, is_color):
    total = len(frame_paths)
    if total <= 0:
        raise ValueError("Kare listesi boş")
    frame_positions = np.linspace(0, total - 1, n_frames).round().astype(int)
    x1, y1, x2, y2 = [int(round(v)) for v in crop_box]
    out_frames, last_ok_frame = [], None
    for pos in frame_positions:
        frame = cv2.imread(str(frame_paths[int(pos)]))
        if frame is None:
            if last_ok_frame is None:
                raise ValueError(f"Hiç geçerli kare okunamadı: {frame_paths[int(pos)]}")
            frame = last_ok_frame.copy()
        else:
            last_ok_frame = frame
        h, w = frame.shape[:2]
        cx1, cy1 = max(0, min(x1, w-1)), max(0, min(y1, h-1))
        cx2, cy2 = max(cx1+1, min(x2, w)), max(cy1+1, min(y2, h))
        cropped = frame[cy1:cy2, cx1:cx2]
        if is_color:
            resized = cv2.resize(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB), (size, size), interpolation=cv2.INTER_AREA)
        else:
            resized = cv2.resize(cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY), (size, size), interpolation=cv2.INTER_AREA)
        out_frames.append(resized)
    return np.stack(out_frames)

bank_ir, bank_depth, failed_keys = {}, {}, []
start = time.time()
for i, it in enumerate(items, start=1):
    key = it["key"]
    try:
        crop_ir = get_crop_box_ir(key)
        crop_depth = get_crop_box_depth(key)
        bank_ir[key] = decode_clip_cropped(it["ir_frames"], crop_ir, CACHE_FRAMES, CACHE_IMAGE_SIZE, is_color=False)
        bank_depth[key] = decode_clip_cropped(it["depth_frames"], crop_depth, CACHE_FRAMES, CACHE_IMAGE_SIZE, is_color=True)
    except Exception as e:
        failed_keys.append((key, str(e)))
        print(f"  !! {key}: cache edilemedi -- {e}")
    if i % 200 == 0 or i == len(items):
        print(f"  [{i}/{len(items)}] cache'lendi ({(time.time()-start)/60:.1f} dk)")

print(f"\n✓ RAM cache: {len(bank_ir)} başarılı, {len(failed_keys)} başarısız.")
if failed_keys:
    failed_key_set = {k for k, _ in failed_keys}
    kept_idx = [i for i, it in enumerate(items) if it["key"] not in failed_key_set]
    items = [items[i] for i in kept_idx]
    old_to_new = {old: new for new, old in enumerate(kept_idx)}
    train_ids = [old_to_new[i] for i in train_ids if i in old_to_new]
    val_ids = [old_to_new[i] for i in val_ids if i in old_to_new]
    print(f"✓ Temizlendi. Yeni: items={len(items)} train={len(train_ids)} val={len(val_ids)}")


# ------------------------------------------------------------------
# 3.2 — DATASET (shared across all architectures; resizing to each
# architecture's native resolution happens inside its own cell, in the
# model's forward pass)
# ------------------------------------------------------------------

stage("3.2 - DATASET")

N_FRAMES_TO_BLACKOUT = 3
NUM_WORKERS = 0

class GateDataset(Dataset):
    def __init__(self, ids, augment=False):
        self.ids = list(ids); self.augment = augment
    def __len__(self): return len(self.ids)
    def __getitem__(self, j):
        idx = self.ids[j]; it = items[idx]
        ir_clip = bank_ir[it["key"]]; depth_clip = bank_depth[it["key"]]
        n = ir_clip.shape[0]
        frame_idx = np.linspace(0, n - 1, TRAIN_FRAMES).round().astype(int)
        ir_frames = ir_clip[frame_idx].copy()
        depth_frames_raw = depth_clip[frame_idx].copy()
        depth_gray = np.stack([cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in depth_frames_raw])
        if self.augment:
            blackout = np.random.choice(TRAIN_FRAMES, size=N_FRAMES_TO_BLACKOUT, replace=False)
            ir_frames[blackout] = 0; depth_gray[blackout] = 0
        ir_rgb = np.stack([np.stack([f, f, f], axis=-1) for f in ir_frames])
        depth_rgb = np.stack([np.stack([f, f, f], axis=-1) for f in depth_gray])
        ir_vis = torch.from_numpy(ir_rgb).permute(3, 0, 1, 2).float() / 255.0
        depth_vis = torch.from_numpy(depth_rgb).permute(3, 0, 1, 2).float() / 255.0
        subj = subject_to_id.get(it["user"], -1)
        return {"ir": ir_vis, "depth": depth_vis, "label": torch.tensor(it["label"]),
                "subject": torch.tensor(subj), "index": torch.tensor(idx)}

def make_loaders(batch_size):
    """Called from each architecture's own cell with its own batch_size --
    so every architecture truly gets its own DataLoader."""
    train_loader = DataLoader(GateDataset(train_ids, augment=True), batch_size=batch_size,
                               shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    train_eval_loader = DataLoader(GateDataset(train_ids, augment=False), batch_size=batch_size,
                                    shuffle=False, num_workers=NUM_WORKERS)
    val_loader = DataLoader(GateDataset(val_ids, augment=False), batch_size=batch_size,
                             shuffle=False, num_workers=NUM_WORKERS)
    return train_loader, train_eval_loader, val_loader

print("✓ GateDataset / make_loaders hazır.")


# ------------------------------------------------------------------
# 3.3 — MODEL PARTS (each architecture sets its own resize_to / lr /
# warmup in its own cell and instantiates these classes)
# ------------------------------------------------------------------

stage("3.3 - MODEL PARÇALARI")

class BackboneWrapper(nn.Module):
    """resize_to is REQUIRED -- each architecture is resized to its own
    native Kinetics resolution (R-family=112, S3D/MViT/Swin3D=224), a single
    resolution is not forced onto all of them."""
    def __init__(self, ctor, weights, resize_to):
        super().__init__()
        m = ctor(weights=weights)
        if hasattr(m, "fc"):
            self.feat_dim = m.fc.in_features; m.fc = nn.Identity()
        elif hasattr(m, "head"):
            head = m.head
            if isinstance(head, nn.Linear):
                self.feat_dim = head.in_features
            else:
                lin = [l for l in head if isinstance(l, nn.Linear)][-1]
                self.feat_dim = lin.in_features
            m.head = nn.Identity()
        elif hasattr(m, "classifier"):
            conv_or_lin = [l for l in m.classifier if isinstance(l, (nn.Conv3d, nn.Linear))][0]
            self.feat_dim = conv_or_lin.in_channels if isinstance(conv_or_lin, nn.Conv3d) else conv_or_lin.in_features
            m.classifier = nn.Identity()
        else:
            raise ValueError(f"Bilinmeyen head yapısı: {type(m)}")
        if hasattr(m, "avgpool"):
            m.avgpool = nn.AdaptiveAvgPool3d(1)
        self.backbone = m; self.resize_to = resize_to
    def forward(self, x):
        x = F.interpolate(x, size=(x.shape[2], self.resize_to, self.resize_to),
                           mode="trilinear", align_corners=False)
        out = self.backbone(x)
        if out.dim() == 5:
            out = torch.flatten(F.adaptive_avg_pool3d(out, 1), 1)
        elif out.dim() == 4:
            out = torch.flatten(F.adaptive_avg_pool2d(out, 1), 1)
        return out

class FeatureLinearGate(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.lin = nn.Linear(dim, dim, bias=False)
        nn.init.zeros_(self.lin.weight)
    def forward(self, feat_own, feat_other):
        return feat_own + self.lin(feat_other)

class SingleModel(nn.Module):
    def __init__(self, ctor, weights, resize_to, n_classes):
        super().__init__()
        self.backbone = BackboneWrapper(ctor, weights, resize_to)
        self.head = nn.Linear(self.backbone.feat_dim, n_classes)
    def forward(self, x): return self.head(self.backbone(x))

class GatedModel(nn.Module):
    def __init__(self, ctor, weights, resize_to, n_classes):
        super().__init__()
        self.backbone_ir = BackboneWrapper(ctor, weights, resize_to)
        self.backbone_depth = BackboneWrapper(ctor, weights, resize_to)
        feat_dim = self.backbone_ir.feat_dim
        self.gate_to_ir = FeatureLinearGate(feat_dim)
        self.gate_to_depth = FeatureLinearGate(feat_dim)
        self.head_ir = nn.Linear(feat_dim, n_classes)
        self.head_depth = nn.Linear(feat_dim, n_classes)
    def forward(self, x_ir, x_depth):
        f_ir = self.backbone_ir(x_ir); f_depth = self.backbone_depth(x_depth)
        f_ir_g = self.gate_to_ir(f_ir, f_depth); f_depth_g = self.gate_to_depth(f_depth, f_ir)
        return self.head_ir(f_ir_g), self.head_depth(f_depth_g)

def make_warmup_cosine(optimizer, total_steps, warmup_frac):
    warmup_steps = max(int(total_steps * warmup_frac), 1)
    def lr_lambda(step):
        if step < warmup_steps: return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print("✓ BackboneWrapper / FeatureLinearGate / SingleModel / GatedModel hazır.")


# ------------------------------------------------------------------
# 3.4 — DRIVE CHECKPOINTING + TRAINING FUNCTIONS (lr/warmup are passed
# in as PARAMETERS from each architecture's own cell). Paths (CKPT_DIR,
# PRED_DIR, ORACLE_DIR, PROGRESS_PATH) come from the path-config cell.
# ------------------------------------------------------------------

stage("3.4 - EĞİTİM ALTYAPISI")

DEVICE = "cuda"
EPOCHS_PER_STAGE = 25

def log(msg): print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

def gpu_cleanup():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    if torch.cuda.is_available():
        log(f"GPU temizlendi -- allocated={torch.cuda.memory_allocated()/1024**3:.1f}GB "
            f"reserved={torch.cuda.memory_reserved()/1024**3:.1f}GB")

progress = {}   # -- intentionally starts clean, no old progress.json is read --
def save_progress(p):
    with open(PROGRESS_PATH, "w") as f: json.dump(p, f, indent=2)

@torch.no_grad()
def evaluate_independent(model_ir, model_depth, loader):
    model_ir.eval(); model_depth.eval()
    c_ir = c_depth = total = 0
    for batch in loader:
        x_ir = batch["ir"].to(DEVICE); x_depth = batch["depth"].to(DEVICE); y = batch["label"].to(DEVICE)
        c_ir += (model_ir(x_ir).argmax(1) == y).sum().item()
        c_depth += (model_depth(x_depth).argmax(1) == y).sum().item()
        total += y.size(0)
    return c_ir/total, c_depth/total

@torch.no_grad()
def evaluate_gated(model, loader):
    model.eval(); c_ir = c_depth = total = 0
    for batch in loader:
        x_ir = batch["ir"].to(DEVICE); x_depth = batch["depth"].to(DEVICE); y = batch["label"].to(DEVICE)
        out_ir, out_depth = model(x_ir, x_depth)
        c_ir += (out_ir.argmax(1) == y).sum().item(); c_depth += (out_depth.argmax(1) == y).sum().item()
        total += y.size(0)
    return c_ir/total, c_depth/total

@torch.no_grad()
def dump_predictions_stage1(model_ir, model_depth, loader, save_path):
    model_ir.eval(); model_depth.eval()
    idxs, labels, probs_ir, probs_depth = [], [], [], []
    for batch in loader:
        x_ir = batch["ir"].to(DEVICE); x_depth = batch["depth"].to(DEVICE)
        probs_ir.append(F.softmax(model_ir(x_ir), 1).cpu().numpy().astype(np.float32))
        probs_depth.append(F.softmax(model_depth(x_depth), 1).cpu().numpy().astype(np.float32))
        labels.append(batch["label"].numpy()); idxs.append(batch["index"].numpy())
    probs_ir, probs_depth = np.concatenate(probs_ir), np.concatenate(probs_depth)
    np.savez(save_path, index=np.concatenate(idxs), label=np.concatenate(labels),
             probs_ir=probs_ir, probs_depth=probs_depth,
             pred_ir=probs_ir.argmax(1), pred_depth=probs_depth.argmax(1))
    log(f"Stage1 tam softmax (N={probs_ir.shape[0]}, 40 sınıf) -> {save_path}")

@torch.no_grad()
def dump_predictions_gated(model, loader, save_path):
    model.eval()
    idxs, labels, probs_ir, probs_depth = [], [], [], []
    for batch in loader:
        x_ir = batch["ir"].to(DEVICE); x_depth = batch["depth"].to(DEVICE)
        out_ir, out_depth = model(x_ir, x_depth)
        probs_ir.append(F.softmax(out_ir, 1).cpu().numpy().astype(np.float32))
        probs_depth.append(F.softmax(out_depth, 1).cpu().numpy().astype(np.float32))
        labels.append(batch["label"].numpy()); idxs.append(batch["index"].numpy())
    probs_ir, probs_depth = np.concatenate(probs_ir), np.concatenate(probs_depth)
    np.savez(save_path, index=np.concatenate(idxs), label=np.concatenate(labels),
             probs_ir=probs_ir, probs_depth=probs_depth,
             pred_ir=probs_ir.argmax(1), pred_depth=probs_depth.argmax(1))
    log(f"Stage3-gated tam softmax (N={probs_ir.shape[0]}, 40 sınıf) -> {save_path}")

def train_stage1(name, ctor, weights, resize_to, train_loader, train_eval_loader, val_loader, backbone_lr, warmup_frac):
    model_ir = SingleModel(ctor, weights, resize_to, N_CLASSES).to(DEVICE)
    model_depth = SingleModel(ctor, weights, resize_to, N_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(list(model_ir.parameters()) + list(model_depth.parameters()), lr=backbone_lr)
    total_steps = EPOCHS_PER_STAGE * len(train_loader)
    scheduler = make_warmup_cosine(optimizer, total_steps, warmup_frac)
    best_combined, best_ir_ckpt, best_depth_ckpt = -1.0, None, None
    for epoch in range(EPOCHS_PER_STAGE):
        model_ir.train(); model_depth.train()
        for batch in train_loader:
            x_ir = batch["ir"].to(DEVICE); x_depth = batch["depth"].to(DEVICE); y = batch["label"].to(DEVICE)
            optimizer.zero_grad()
            loss = F.cross_entropy(model_ir(x_ir), y) + F.cross_entropy(model_depth(x_depth), y)
            loss.backward(); optimizer.step(); scheduler.step()
        tr_ir, tr_depth = evaluate_independent(model_ir, model_depth, train_eval_loader)
        val_ir, val_depth = evaluate_independent(model_ir, model_depth, val_loader)
        combined = (val_ir + val_depth) / 2
        tag = ""
        if combined > best_combined:
            best_combined = combined
            # -- copy onto CPU: don't let repeated deepcopy on GPU pile up
            # extra full model copies and inflate reserved memory --
            best_ir_ckpt = {"backbone": {k: v.detach().cpu().clone() for k, v in model_ir.backbone.state_dict().items()},
                             "head": {k: v.detach().cpu().clone() for k, v in model_ir.head.state_dict().items()}}
            best_depth_ckpt = {"backbone": {k: v.detach().cpu().clone() for k, v in model_depth.backbone.state_dict().items()},
                                "head": {k: v.detach().cpu().clone() for k, v in model_depth.head.state_dict().items()}}
            torch.save(best_ir_ckpt, f"{CKPT_DIR}/{name}_stage1_ir.pt")
            torch.save(best_depth_ckpt, f"{CKPT_DIR}/{name}_stage1_depth.pt")
            tag = "  <-- yeni en iyi (Drive'a kaydedildi)"
        log(f"[{name} S1] e{epoch+1}/{EPOCHS_PER_STAGE} | TRAIN IR={tr_ir:.4f} Depth={tr_depth:.4f} "
            f"| VAL IR={val_ir:.4f} Depth={val_depth:.4f} (ort={combined:.4f}){tag}")
    model_ir.backbone.load_state_dict(best_ir_ckpt["backbone"]); model_ir.head.load_state_dict(best_ir_ckpt["head"])
    model_depth.backbone.load_state_dict(best_depth_ckpt["backbone"]); model_depth.head.load_state_dict(best_depth_ckpt["head"])
    dump_predictions_stage1(model_ir, model_depth, val_loader, f"{PRED_DIR}/{name}_stage1_predictions.npz")
    del optimizer, scheduler
    return model_ir, model_depth, best_combined

def train_gated_stage(gated_model, train_loader, train_eval_loader, val_loader, stage_label, arch_name,
                       freeze_backbone, backbone_lr, gate_lr, warmup_frac):
    if freeze_backbone:
        for p in gated_model.backbone_ir.parameters(): p.requires_grad = False
        for p in gated_model.backbone_depth.parameters(): p.requires_grad = False
        params_ir = list(gated_model.gate_to_ir.parameters()) + list(gated_model.head_ir.parameters())
        params_depth = list(gated_model.gate_to_depth.parameters()) + list(gated_model.head_depth.parameters())
        lr = gate_lr
    else:
        for p in gated_model.backbone_ir.parameters(): p.requires_grad = True
        for p in gated_model.backbone_depth.parameters(): p.requires_grad = True
        params_ir = list(gated_model.backbone_ir.parameters()) + list(gated_model.gate_to_ir.parameters()) + list(gated_model.head_ir.parameters())
        params_depth = list(gated_model.backbone_depth.parameters()) + list(gated_model.gate_to_depth.parameters()) + list(gated_model.head_depth.parameters())
        lr = backbone_lr
    opt_ir = torch.optim.AdamW(params_ir, lr=lr)
    opt_depth = torch.optim.AdamW(params_depth, lr=lr)
    total_steps = EPOCHS_PER_STAGE * len(train_loader)
    sched_ir = make_warmup_cosine(opt_ir, total_steps, warmup_frac)
    sched_depth = make_warmup_cosine(opt_depth, total_steps, warmup_frac)
    best_combined, best_state = -1.0, None
    for epoch in range(EPOCHS_PER_STAGE):
        gated_model.train()
        if freeze_backbone:
            # backbone frozen -> keep BatchNorm running mean/var from drifting too
            gated_model.backbone_ir.eval()
            gated_model.backbone_depth.eval()
        for batch in train_loader:
            x_ir = batch["ir"].to(DEVICE); x_depth = batch["depth"].to(DEVICE); y = batch["label"].to(DEVICE)
            out_ir, out_depth = gated_model(x_ir, x_depth)
            loss_ir = F.cross_entropy(out_ir, y); loss_depth = F.cross_entropy(out_depth, y)
            opt_ir.zero_grad(); opt_depth.zero_grad()
            loss_ir.backward(retain_graph=True); loss_depth.backward()
            opt_ir.step(); opt_depth.step(); sched_ir.step(); sched_depth.step()
        tr_ir, tr_depth = evaluate_gated(gated_model, train_eval_loader)
        val_ir, val_depth = evaluate_gated(gated_model, val_loader)
        combined = (val_ir + val_depth) / 2
        tag = ""
        if combined > best_combined:
            best_combined = combined
            # -- copy onto CPU (don't let GPU memory bloat) --
            best_state = {k: v.detach().cpu().clone() for k, v in gated_model.state_dict().items()}
            torch.save(best_state, f"{CKPT_DIR}/{arch_name}_{stage_label}_full.pt")
            tag = "  <-- yeni en iyi (Drive'a kaydedildi)"
        log(f"[{arch_name} {stage_label}] e{epoch+1}/{EPOCHS_PER_STAGE} | TRAIN IR={tr_ir:.4f} Depth={tr_depth:.4f} "
            f"| VAL IR={val_ir:.4f} Depth={val_depth:.4f} (ort={combined:.4f}){tag}")
    gated_model.load_state_dict(best_state)
    del opt_ir, opt_depth, sched_ir, sched_depth
    return gated_model, best_combined

print("✓ train_stage1 / train_gated_stage hazır (lr ve warmup her mimarinin "
      "kendi hücresinden geçiriliyor).")


# ------------------------------------------------------------------
# 3.5 — ORACLE ANALYSIS (over the full 40-class softmax) -- each
# architecture's own training cell will call this once its model is done
# ------------------------------------------------------------------

stage("3.5 - ORACLE ANALİZİ FONKSİYONLARI")

def _entropy(probs):
    eps = 1e-12; return -(probs * np.log(probs + eps)).sum(axis=1)

def _top2_margin(probs):
    s = np.sort(probs, axis=1); return s[:, -1] - s[:, -2]

def oracle_analysis(arch_name):
    s1_path = f"{PRED_DIR}/{arch_name}_stage1_predictions.npz"
    s3_path = f"{PRED_DIR}/{arch_name}_stage3_predictions.npz"
    if not (os.path.exists(s1_path) and os.path.exists(s3_path)):
        log(f"[{arch_name}] prediction dosyaları eksik, atlanıyor."); return None
    s1, s3 = np.load(s1_path), np.load(s3_path)
    order1, order3 = np.argsort(s1["index"]), np.argsort(s3["index"])
    label = s1["label"][order1]
    assert np.array_equal(label, s3["label"][order3]), "index/label sırası uyuşmuyor"
    n = len(label)
    probs_ir_s1, probs_depth_s1 = s1["probs_ir"][order1], s1["probs_depth"][order1]
    probs_ir_s3, probs_depth_s3 = s3["probs_ir"][order3], s3["probs_depth"][order3]
    pred_ir_s1, pred_depth_s1 = probs_ir_s1.argmax(1), probs_depth_s1.argmax(1)
    pred_ir_s3, pred_depth_s3 = probs_ir_s3.argmax(1), probs_depth_s3.argmax(1)

    # ---- BREAKDOWN 1: GATE-FREE (Stage1 -- independently/parallel-trained IR and Depth models) ----
    ir_baseline_correct = pred_ir_s1 == label
    depth_baseline_correct = pred_depth_s1 == label
    oracle_correct_gatesiz = ir_baseline_correct | depth_baseline_correct
    oracle_acc_gatesiz = float(oracle_correct_gatesiz.mean())

    # ---- BREAKDOWN 2: STAGE 3 (gated -- IR and Depth after end-to-end training) ----
    ir_gated_correct = pred_ir_s3 == label
    depth_gated_correct = pred_depth_s3 == label
    oracle_correct_stage3 = ir_gated_correct | depth_gated_correct
    oracle_acc_stage3 = float(oracle_correct_stage3.mean())
    oracle_gain = oracle_acc_stage3 - oracle_acc_gatesiz

    # ---- Depth's share from the gate (redundant/complementary breakdown relative to the IR baseline) ----
    newly_fixed = (~depth_baseline_correct) & depth_gated_correct
    newly_broken = depth_baseline_correct & (~depth_gated_correct)
    redundant = newly_fixed & ir_baseline_correct
    complementary = newly_fixed & (~ir_baseline_correct)
    true_conf_s1 = probs_depth_s1[np.arange(n), label]; true_conf_s3 = probs_depth_s3[np.arange(n), label]
    margin_s1, margin_s3 = _top2_margin(probs_depth_s1), _top2_margin(probs_depth_s3)
    entropy_s1, entropy_s3 = _entropy(probs_depth_s1), _entropy(probs_depth_s3)

    log(f"\n--- Oracle: {arch_name} ---")
    log(f"[Detay 1 -- GATESİZ / Stage1] IR acc={ir_baseline_correct.mean():.4f}  Depth acc={depth_baseline_correct.mean():.4f}  "
        f"Oracle (IR-veya-Depth doğruysa doğru sayılır)={oracle_acc_gatesiz:.4f}")
    log(f"[Detay 2 -- STAGE 3 / gate'li] IR acc={ir_gated_correct.mean():.4f}  Depth acc={depth_gated_correct.mean():.4f}  "
        f"Oracle (IR-veya-Depth doğruysa doğru sayılır)={oracle_acc_stage3:.4f}")
    log(f"Oracle kazancı (gate'in IR+Depth arası tamamlayıcılığa katkısı): "
        f"{'+' if oracle_gain >= 0 else ''}{oracle_gain*100:.2f} puan")
    log(f"Depth kendi içinde: baseline={depth_baseline_correct.mean():.4f} -> gated={depth_gated_correct.mean():.4f}  |  "
        f"Yeni doğru={newly_fixed.sum()} ({newly_fixed.mean()*100:.1f}%)  "
        f"redundant={redundant.sum()}  complementary={complementary.sum()}  regresyon={newly_broken.sum()}")

    df = pd.DataFrame({
        "index": s1["index"][order1], "label": label,
        "pred_ir_baseline": pred_ir_s1, "pred_depth_baseline": pred_depth_s1,
        "pred_ir_gated": pred_ir_s3, "pred_depth_gated": pred_depth_s3,
        "ir_baseline_correct": ir_baseline_correct, "depth_baseline_correct": depth_baseline_correct,
        "ir_gated_correct": ir_gated_correct, "depth_gated_correct": depth_gated_correct,
        "oracle_correct_gatesiz": oracle_correct_gatesiz, "oracle_correct_stage3": oracle_correct_stage3,
        "newly_fixed": newly_fixed, "redundant": redundant, "complementary": complementary, "newly_broken": newly_broken,
        "true_class_conf_depth_baseline": true_conf_s1, "true_class_conf_depth_gated": true_conf_s3,
        "margin_depth_baseline": margin_s1, "margin_depth_gated": margin_s3,
        "entropy_depth_baseline": entropy_s1, "entropy_depth_gated": entropy_s3,
    })
    df.to_csv(f"{ORACLE_DIR}/{arch_name}_oracle_detail.csv", index=False)
    summary = {
        "detay1_gatesiz": {
            "ir_acc": float(ir_baseline_correct.mean()),
            "depth_acc": float(depth_baseline_correct.mean()),
            "oracle_acc": oracle_acc_gatesiz,
        },
        "detay2_stage3": {
            "ir_acc": float(ir_gated_correct.mean()),
            "depth_acc": float(depth_gated_correct.mean()),
            "oracle_acc": oracle_acc_stage3,
        },
        "oracle_gain": oracle_gain,
        # -- the old flat fields are also kept (so Cell 16 can still read them backward-compatibly) --
        "depth_baseline_acc": float(depth_baseline_correct.mean()), "depth_gated_acc": float(depth_gated_correct.mean()),
        "newly_fixed": int(newly_fixed.sum()), "redundant": int(redundant.sum()),
        "complementary": int(complementary.sum()), "newly_broken": int(newly_broken.sum()),
        "mean_true_class_conf_baseline": float(true_conf_s1.mean()), "mean_true_class_conf_gated": float(true_conf_s3.mean()),
        "mean_margin_baseline": float(margin_s1.mean()), "mean_margin_gated": float(margin_s3.mean()),
        "mean_entropy_baseline": float(entropy_s1.mean()), "mean_entropy_gated": float(entropy_s3.mean()),
    }
    with open(f"{ORACLE_DIR}/{arch_name}_oracle_summary.json", "w") as f: json.dump(summary, f, indent=2)
    return summary

print("✓ Hücre 3 tamamlandı: bank_ir, bank_depth, GateDataset, model sınıfları, "
      "eğitim fonksiyonları, oracle_analysis() hazır. Şimdi sırayla 6 mimarinin "
      "kendi hücrelerine geçebilirsin.")


In [ ]:
# ============================================================
# CELL 4/20 — r2plus1d_18 DATA LOADER (R(2+1)D -- CNN family)
# Pretrained on Kinetics at native 112x112; standard CNN LR/warmup.
# Run after Cell 3 (RAM cache + shared definitions).
# ============================================================

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("04 - r2plus1d_18 VERİ YÜKLEYİCİ")

R2PLUS1D_18_NAME = "r2plus1d_18"
R2PLUS1D_18_CTOR = video_models.r2plus1d_18
R2PLUS1D_18_WEIGHTS = video_models.R2Plus1D_18_Weights.KINETICS400_V1
R2PLUS1D_18_RESIZE_TO = 112      # native Kinetics resolution
R2PLUS1D_18_BATCH_SIZE = 64
R2PLUS1D_18_BACKBONE_LR = 1e-4
R2PLUS1D_18_GATE_LR = 2e-3
R2PLUS1D_18_WARMUP_FRAC = 0.05

R2PLUS1D_18_train_loader, R2PLUS1D_18_train_eval_loader, R2PLUS1D_18_val_loader = make_loaders(R2PLUS1D_18_BATCH_SIZE)

print(f"✓ {R2PLUS1D_18_NAME} data loader hazır: resize_to={R2PLUS1D_18_RESIZE_TO} "
      f"batch_size={R2PLUS1D_18_BATCH_SIZE} backbone_lr={R2PLUS1D_18_BACKBONE_LR} "
      f"warmup_frac={R2PLUS1D_18_WARMUP_FRAC}")
print(f"  train_batches={len(R2PLUS1D_18_train_loader)} val_batches={len(R2PLUS1D_18_val_loader)}")

In [ ]:
# ============================================================
# CELL 5/20 — r2plus1d_18: 3-STAGE TRAINING + ORACLE ANALYSIS
# Run after Cell 4 (its own loader must be ready).
# Stage1(25) -> Stage2(25, gate-only) -> Stage3(25, end-to-end)
# ============================================================

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("05 - r2plus1d_18 EĞİTİMİ")

progress.setdefault(R2PLUS1D_18_NAME, {})
R2PLUS1D_18_batch_size = R2PLUS1D_18_BATCH_SIZE
R2PLUS1D_18_done = False

while not R2PLUS1D_18_done:
    try:
        gpu_cleanup()

        # ---- STAGE 1: independent, parallel, one loss + one optimizer each ----
        R2PLUS1D_18_model_ir, R2PLUS1D_18_model_depth, R2PLUS1D_18_s1_acc = train_stage1(
            R2PLUS1D_18_NAME, R2PLUS1D_18_CTOR, R2PLUS1D_18_WEIGHTS, R2PLUS1D_18_RESIZE_TO,
            R2PLUS1D_18_train_loader, R2PLUS1D_18_train_eval_loader, R2PLUS1D_18_val_loader,
            backbone_lr=R2PLUS1D_18_BACKBONE_LR, warmup_frac=R2PLUS1D_18_WARMUP_FRAC,
        )
        progress[R2PLUS1D_18_NAME]["stage1_acc"] = R2PLUS1D_18_s1_acc
        save_progress(progress)
        del R2PLUS1D_18_model_ir, R2PLUS1D_18_model_depth
        gpu_cleanup()

        # ---- GATED MODEL, initialized from Stage1 weights ----
        R2PLUS1D_18_gated = GatedModel(R2PLUS1D_18_CTOR, R2PLUS1D_18_WEIGHTS, R2PLUS1D_18_RESIZE_TO, N_CLASSES).to(DEVICE)
        R2PLUS1D_18_ck_ir = torch.load(f"{CKPT_DIR}/{R2PLUS1D_18_NAME}_stage1_ir.pt", map_location="cpu")
        R2PLUS1D_18_ck_depth = torch.load(f"{CKPT_DIR}/{R2PLUS1D_18_NAME}_stage1_depth.pt", map_location="cpu")
        R2PLUS1D_18_gated.backbone_ir.load_state_dict(R2PLUS1D_18_ck_ir["backbone"])
        R2PLUS1D_18_gated.backbone_depth.load_state_dict(R2PLUS1D_18_ck_depth["backbone"])
        R2PLUS1D_18_gated.head_ir.load_state_dict(R2PLUS1D_18_ck_ir["head"])
        R2PLUS1D_18_gated.head_depth.load_state_dict(R2PLUS1D_18_ck_depth["head"])

        # ---- STAGE 2: gate-only, backbone frozen ----
        R2PLUS1D_18_gated, R2PLUS1D_18_s2_acc = train_gated_stage(
            R2PLUS1D_18_gated, R2PLUS1D_18_train_loader, R2PLUS1D_18_train_eval_loader, R2PLUS1D_18_val_loader,
            "stage2", R2PLUS1D_18_NAME, freeze_backbone=True,
            backbone_lr=R2PLUS1D_18_BACKBONE_LR, gate_lr=R2PLUS1D_18_GATE_LR, warmup_frac=R2PLUS1D_18_WARMUP_FRAC,
        )
        progress[R2PLUS1D_18_NAME]["stage2_acc"] = R2PLUS1D_18_s2_acc
        save_progress(progress)
        gpu_cleanup()   # GPU cleanup between Stage2 and Stage3

        # ---- STAGE 3: end-to-end ----
        R2PLUS1D_18_gated, R2PLUS1D_18_s3_acc = train_gated_stage(
            R2PLUS1D_18_gated, R2PLUS1D_18_train_loader, R2PLUS1D_18_train_eval_loader, R2PLUS1D_18_val_loader,
            "stage3", R2PLUS1D_18_NAME, freeze_backbone=False,
            backbone_lr=R2PLUS1D_18_BACKBONE_LR, gate_lr=R2PLUS1D_18_GATE_LR, warmup_frac=R2PLUS1D_18_WARMUP_FRAC,
        )
        dump_predictions_gated(R2PLUS1D_18_gated, R2PLUS1D_18_val_loader, f"{PRED_DIR}/{R2PLUS1D_18_NAME}_stage3_predictions.npz")
        progress[R2PLUS1D_18_NAME]["stage3_acc"] = R2PLUS1D_18_s3_acc
        progress[R2PLUS1D_18_NAME]["done"] = True
        save_progress(progress)

        log(f"[{R2PLUS1D_18_NAME}] TAMAMLANDI: Stage1={R2PLUS1D_18_s1_acc:.4f} "
            f"Stage2={R2PLUS1D_18_s2_acc:.4f} Stage3-E2E={R2PLUS1D_18_s3_acc:.4f}")

        del R2PLUS1D_18_gated
        gpu_cleanup()
        R2PLUS1D_18_done = True

    except torch.cuda.OutOfMemoryError:
        gpu_cleanup()
        R2PLUS1D_18_batch_size = max(R2PLUS1D_18_batch_size // 2, 4)
        log(f"[{R2PLUS1D_18_NAME}] !! OOM -- batch_size={R2PLUS1D_18_batch_size} ile tekrar deneniyor")
        R2PLUS1D_18_train_loader, R2PLUS1D_18_train_eval_loader, R2PLUS1D_18_val_loader = make_loaders(R2PLUS1D_18_batch_size)

    except Exception as e:
        log(f"[{R2PLUS1D_18_NAME}] HATA: {repr(e)}")
        progress[R2PLUS1D_18_NAME]["error"] = str(e)
        save_progress(progress)
        raise

# ---- Oracle analysis (for this architecture) ----
R2PLUS1D_18_oracle_summary = oracle_analysis(R2PLUS1D_18_NAME)

print(f"\n✓ {R2PLUS1D_18_NAME} tamamlandı ve oracle analizi kaydedildi -> {ORACLE_DIR}")

In [ ]:
# ============================================================
# CELL 6/20 — r3d_18 DATA LOADER (R3D -- CNN family)
# Pretrained on Kinetics at native 112x112; standard CNN LR/warmup.
# Run after Cell 3 (RAM cache + shared definitions).
# ============================================================

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("06 - r3d_18 VERİ YÜKLEYİCİ")

R3D_18_NAME = "r3d_18"
R3D_18_CTOR = video_models.r3d_18
R3D_18_WEIGHTS = video_models.R3D_18_Weights.KINETICS400_V1
R3D_18_RESIZE_TO = 112      # native Kinetics resolution
R3D_18_BATCH_SIZE = 64
R3D_18_BACKBONE_LR = 1e-4
R3D_18_GATE_LR = 2e-3
R3D_18_WARMUP_FRAC = 0.05

R3D_18_train_loader, R3D_18_train_eval_loader, R3D_18_val_loader = make_loaders(R3D_18_BATCH_SIZE)

print(f"✓ {R3D_18_NAME} data loader hazır: resize_to={R3D_18_RESIZE_TO} "
      f"batch_size={R3D_18_BATCH_SIZE} backbone_lr={R3D_18_BACKBONE_LR} "
      f"warmup_frac={R3D_18_WARMUP_FRAC}")
print(f"  train_batches={len(R3D_18_train_loader)} val_batches={len(R3D_18_val_loader)}")

In [ ]:
# ============================================================
# CELL 7/20 — r3d_18: 3-STAGE TRAINING + ORACLE ANALYSIS
# Run after Cell 6 (its own loader must be ready).
# Stage1(25) -> Stage2(25, gate-only) -> Stage3(25, end-to-end)
# ============================================================

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("07 - r3d_18 EĞİTİMİ")

progress.setdefault(R3D_18_NAME, {})
R3D_18_batch_size = R3D_18_BATCH_SIZE
R3D_18_done = False

while not R3D_18_done:
    try:
        gpu_cleanup()

        # ---- STAGE 1: independent, parallel, one loss + one optimizer each ----
        R3D_18_model_ir, R3D_18_model_depth, R3D_18_s1_acc = train_stage1(
            R3D_18_NAME, R3D_18_CTOR, R3D_18_WEIGHTS, R3D_18_RESIZE_TO,
            R3D_18_train_loader, R3D_18_train_eval_loader, R3D_18_val_loader,
            backbone_lr=R3D_18_BACKBONE_LR, warmup_frac=R3D_18_WARMUP_FRAC,
        )
        progress[R3D_18_NAME]["stage1_acc"] = R3D_18_s1_acc
        save_progress(progress)
        del R3D_18_model_ir, R3D_18_model_depth
        gpu_cleanup()

        # ---- GATED MODEL, initialized from Stage1 weights ----
        R3D_18_gated = GatedModel(R3D_18_CTOR, R3D_18_WEIGHTS, R3D_18_RESIZE_TO, N_CLASSES).to(DEVICE)
        R3D_18_ck_ir = torch.load(f"{CKPT_DIR}/{R3D_18_NAME}_stage1_ir.pt", map_location="cpu")
        R3D_18_ck_depth = torch.load(f"{CKPT_DIR}/{R3D_18_NAME}_stage1_depth.pt", map_location="cpu")
        R3D_18_gated.backbone_ir.load_state_dict(R3D_18_ck_ir["backbone"])
        R3D_18_gated.backbone_depth.load_state_dict(R3D_18_ck_depth["backbone"])
        R3D_18_gated.head_ir.load_state_dict(R3D_18_ck_ir["head"])
        R3D_18_gated.head_depth.load_state_dict(R3D_18_ck_depth["head"])

        # ---- STAGE 2: gate-only, backbone frozen ----
        R3D_18_gated, R3D_18_s2_acc = train_gated_stage(
            R3D_18_gated, R3D_18_train_loader, R3D_18_train_eval_loader, R3D_18_val_loader,
            "stage2", R3D_18_NAME, freeze_backbone=True,
            backbone_lr=R3D_18_BACKBONE_LR, gate_lr=R3D_18_GATE_LR, warmup_frac=R3D_18_WARMUP_FRAC,
        )
        progress[R3D_18_NAME]["stage2_acc"] = R3D_18_s2_acc
        save_progress(progress)
        gpu_cleanup()   # GPU cleanup between Stage2 and Stage3

        # ---- STAGE 3: end-to-end ----
        R3D_18_gated, R3D_18_s3_acc = train_gated_stage(
            R3D_18_gated, R3D_18_train_loader, R3D_18_train_eval_loader, R3D_18_val_loader,
            "stage3", R3D_18_NAME, freeze_backbone=False,
            backbone_lr=R3D_18_BACKBONE_LR, gate_lr=R3D_18_GATE_LR, warmup_frac=R3D_18_WARMUP_FRAC,
        )
        dump_predictions_gated(R3D_18_gated, R3D_18_val_loader, f"{PRED_DIR}/{R3D_18_NAME}_stage3_predictions.npz")
        progress[R3D_18_NAME]["stage3_acc"] = R3D_18_s3_acc
        progress[R3D_18_NAME]["done"] = True
        save_progress(progress)

        log(f"[{R3D_18_NAME}] TAMAMLANDI: Stage1={R3D_18_s1_acc:.4f} "
            f"Stage2={R3D_18_s2_acc:.4f} Stage3-E2E={R3D_18_s3_acc:.4f}")

        del R3D_18_gated
        gpu_cleanup()
        R3D_18_done = True

    except torch.cuda.OutOfMemoryError:
        gpu_cleanup()
        R3D_18_batch_size = max(R3D_18_batch_size // 2, 4)
        log(f"[{R3D_18_NAME}] !! OOM -- batch_size={R3D_18_batch_size} ile tekrar deneniyor")
        R3D_18_train_loader, R3D_18_train_eval_loader, R3D_18_val_loader = make_loaders(R3D_18_batch_size)

    except Exception as e:
        log(f"[{R3D_18_NAME}] HATA: {repr(e)}")
        progress[R3D_18_NAME]["error"] = str(e)
        save_progress(progress)
        raise

# ---- Oracle analysis (for this architecture) ----
R3D_18_oracle_summary = oracle_analysis(R3D_18_NAME)

print(f"\n✓ {R3D_18_NAME} tamamlandı ve oracle analizi kaydedildi -> {ORACLE_DIR}")

In [ ]:
# ============================================================
# CELL 8/20 — mc3_18 DATA LOADER (MC3 -- CNN family)
# Pretrained on Kinetics at native 112x112; standard CNN LR/warmup.
# Run after Cell 3 (RAM cache + shared definitions).
# ============================================================

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("08 - mc3_18 VERİ YÜKLEYİCİ")

MC3_18_NAME = "mc3_18"
MC3_18_CTOR = video_models.mc3_18
MC3_18_WEIGHTS = video_models.MC3_18_Weights.KINETICS400_V1
MC3_18_RESIZE_TO = 112      # native Kinetics resolution
MC3_18_BATCH_SIZE = 64
MC3_18_BACKBONE_LR = 1e-4
MC3_18_GATE_LR = 2e-3
MC3_18_WARMUP_FRAC = 0.05

MC3_18_train_loader, MC3_18_train_eval_loader, MC3_18_val_loader = make_loaders(MC3_18_BATCH_SIZE)

print(f"✓ {MC3_18_NAME} data loader hazır: resize_to={MC3_18_RESIZE_TO} "
      f"batch_size={MC3_18_BATCH_SIZE} backbone_lr={MC3_18_BACKBONE_LR} "
      f"warmup_frac={MC3_18_WARMUP_FRAC}")
print(f"  train_batches={len(MC3_18_train_loader)} val_batches={len(MC3_18_val_loader)}")

In [ ]:
# ============================================================
# CELL 9/20 — mc3_18: 3-STAGE TRAINING + ORACLE ANALYSIS
# Run after Cell 8 (its own loader must be ready).
# Stage1(25) -> Stage2(25, gate-only) -> Stage3(25, end-to-end)
# ============================================================

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("09 - mc3_18 EĞİTİMİ")

progress.setdefault(MC3_18_NAME, {})
MC3_18_batch_size = MC3_18_BATCH_SIZE
MC3_18_done = False

while not MC3_18_done:
    try:
        gpu_cleanup()

        # ---- STAGE 1: independent, parallel, one loss + one optimizer each ----
        MC3_18_model_ir, MC3_18_model_depth, MC3_18_s1_acc = train_stage1(
            MC3_18_NAME, MC3_18_CTOR, MC3_18_WEIGHTS, MC3_18_RESIZE_TO,
            MC3_18_train_loader, MC3_18_train_eval_loader, MC3_18_val_loader,
            backbone_lr=MC3_18_BACKBONE_LR, warmup_frac=MC3_18_WARMUP_FRAC,
        )
        progress[MC3_18_NAME]["stage1_acc"] = MC3_18_s1_acc
        save_progress(progress)
        del MC3_18_model_ir, MC3_18_model_depth
        gpu_cleanup()

        # ---- GATED MODEL, initialized from Stage1 weights ----
        MC3_18_gated = GatedModel(MC3_18_CTOR, MC3_18_WEIGHTS, MC3_18_RESIZE_TO, N_CLASSES).to(DEVICE)
        MC3_18_ck_ir = torch.load(f"{CKPT_DIR}/{MC3_18_NAME}_stage1_ir.pt", map_location="cpu")
        MC3_18_ck_depth = torch.load(f"{CKPT_DIR}/{MC3_18_NAME}_stage1_depth.pt", map_location="cpu")
        MC3_18_gated.backbone_ir.load_state_dict(MC3_18_ck_ir["backbone"])
        MC3_18_gated.backbone_depth.load_state_dict(MC3_18_ck_depth["backbone"])
        MC3_18_gated.head_ir.load_state_dict(MC3_18_ck_ir["head"])
        MC3_18_gated.head_depth.load_state_dict(MC3_18_ck_depth["head"])

        # ---- STAGE 2: gate-only, backbone frozen ----
        MC3_18_gated, MC3_18_s2_acc = train_gated_stage(
            MC3_18_gated, MC3_18_train_loader, MC3_18_train_eval_loader, MC3_18_val_loader,
            "stage2", MC3_18_NAME, freeze_backbone=True,
            backbone_lr=MC3_18_BACKBONE_LR, gate_lr=MC3_18_GATE_LR, warmup_frac=MC3_18_WARMUP_FRAC,
        )
        progress[MC3_18_NAME]["stage2_acc"] = MC3_18_s2_acc
        save_progress(progress)
        gpu_cleanup()   # GPU cleanup between Stage2 and Stage3

        # ---- STAGE 3: end-to-end ----
        MC3_18_gated, MC3_18_s3_acc = train_gated_stage(
            MC3_18_gated, MC3_18_train_loader, MC3_18_train_eval_loader, MC3_18_val_loader,
            "stage3", MC3_18_NAME, freeze_backbone=False,
            backbone_lr=MC3_18_BACKBONE_LR, gate_lr=MC3_18_GATE_LR, warmup_frac=MC3_18_WARMUP_FRAC,
        )
        dump_predictions_gated(MC3_18_gated, MC3_18_val_loader, f"{PRED_DIR}/{MC3_18_NAME}_stage3_predictions.npz")
        progress[MC3_18_NAME]["stage3_acc"] = MC3_18_s3_acc
        progress[MC3_18_NAME]["done"] = True
        save_progress(progress)

        log(f"[{MC3_18_NAME}] TAMAMLANDI: Stage1={MC3_18_s1_acc:.4f} "
            f"Stage2={MC3_18_s2_acc:.4f} Stage3-E2E={MC3_18_s3_acc:.4f}")

        del MC3_18_gated
        gpu_cleanup()
        MC3_18_done = True

    except torch.cuda.OutOfMemoryError:
        gpu_cleanup()
        MC3_18_batch_size = max(MC3_18_batch_size // 2, 4)
        log(f"[{MC3_18_NAME}] !! OOM -- batch_size={MC3_18_batch_size} ile tekrar deneniyor")
        MC3_18_train_loader, MC3_18_train_eval_loader, MC3_18_val_loader = make_loaders(MC3_18_batch_size)

    except Exception as e:
        log(f"[{MC3_18_NAME}] HATA: {repr(e)}")
        progress[MC3_18_NAME]["error"] = str(e)
        save_progress(progress)
        raise

# ---- Oracle analysis (for this architecture) ----
MC3_18_oracle_summary = oracle_analysis(MC3_18_NAME)

print(f"\n✓ {MC3_18_NAME} tamamlandı ve oracle analizi kaydedildi -> {ORACLE_DIR}")

In [ ]:
# ============================================================
# CELL 10/20 — s3d DATA LOADER (S3D -- Inception/CNN family)
# Pretrained on Kinetics at native 224x224; standard CNN LR/warmup since it's a CNN, but batch size was reduced because of the 224px input and deeper layers.
# Run after Cell 3 (RAM cache + shared definitions).
# ============================================================

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("10 - s3d VERİ YÜKLEYİCİ")

S3D_NAME = "s3d"
S3D_CTOR = video_models.s3d
S3D_WEIGHTS = video_models.S3D_Weights.KINETICS400_V1
S3D_RESIZE_TO = 224      # native Kinetics resolution
S3D_BATCH_SIZE = 32
S3D_BACKBONE_LR = 1e-4
S3D_GATE_LR = 2e-3
S3D_WARMUP_FRAC = 0.05

S3D_train_loader, S3D_train_eval_loader, S3D_val_loader = make_loaders(S3D_BATCH_SIZE)

print(f"✓ {S3D_NAME} data loader hazır: resize_to={S3D_RESIZE_TO} "
      f"batch_size={S3D_BATCH_SIZE} backbone_lr={S3D_BACKBONE_LR} "
      f"warmup_frac={S3D_WARMUP_FRAC}")
print(f"  train_batches={len(S3D_train_loader)} val_batches={len(S3D_val_loader)}")

In [ ]:
# ============================================================
# CELL 11/20 — s3d: 3-STAGE TRAINING + ORACLE ANALYSIS
# Run after Cell 10 (its own loader must be ready).
# Stage1(25) -> Stage2(25, gate-only) -> Stage3(25, end-to-end)
# ============================================================

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("11 - s3d EĞİTİMİ")

progress.setdefault(S3D_NAME, {})
S3D_batch_size = S3D_BATCH_SIZE
S3D_done = False

while not S3D_done:
    try:
        gpu_cleanup()

        # ---- STAGE 1: independent, parallel, one loss + one optimizer each ----
        S3D_model_ir, S3D_model_depth, S3D_s1_acc = train_stage1(
            S3D_NAME, S3D_CTOR, S3D_WEIGHTS, S3D_RESIZE_TO,
            S3D_train_loader, S3D_train_eval_loader, S3D_val_loader,
            backbone_lr=S3D_BACKBONE_LR, warmup_frac=S3D_WARMUP_FRAC,
        )
        progress[S3D_NAME]["stage1_acc"] = S3D_s1_acc
        save_progress(progress)
        del S3D_model_ir, S3D_model_depth
        gpu_cleanup()

        # ---- GATED MODEL, initialized from Stage1 weights ----
        S3D_gated = GatedModel(S3D_CTOR, S3D_WEIGHTS, S3D_RESIZE_TO, N_CLASSES).to(DEVICE)
        S3D_ck_ir = torch.load(f"{CKPT_DIR}/{S3D_NAME}_stage1_ir.pt", map_location="cpu")
        S3D_ck_depth = torch.load(f"{CKPT_DIR}/{S3D_NAME}_stage1_depth.pt", map_location="cpu")
        S3D_gated.backbone_ir.load_state_dict(S3D_ck_ir["backbone"])
        S3D_gated.backbone_depth.load_state_dict(S3D_ck_depth["backbone"])
        S3D_gated.head_ir.load_state_dict(S3D_ck_ir["head"])
        S3D_gated.head_depth.load_state_dict(S3D_ck_depth["head"])

        # ---- STAGE 2: gate-only, backbone frozen ----
        S3D_gated, S3D_s2_acc = train_gated_stage(
            S3D_gated, S3D_train_loader, S3D_train_eval_loader, S3D_val_loader,
            "stage2", S3D_NAME, freeze_backbone=True,
            backbone_lr=S3D_BACKBONE_LR, gate_lr=S3D_GATE_LR, warmup_frac=S3D_WARMUP_FRAC,
        )
        progress[S3D_NAME]["stage2_acc"] = S3D_s2_acc
        save_progress(progress)
        gpu_cleanup()   # GPU cleanup between Stage2 and Stage3

        # ---- STAGE 3: end-to-end ----
        S3D_gated, S3D_s3_acc = train_gated_stage(
            S3D_gated, S3D_train_loader, S3D_train_eval_loader, S3D_val_loader,
            "stage3", S3D_NAME, freeze_backbone=False,
            backbone_lr=S3D_BACKBONE_LR, gate_lr=S3D_GATE_LR, warmup_frac=S3D_WARMUP_FRAC,
        )
        dump_predictions_gated(S3D_gated, S3D_val_loader, f"{PRED_DIR}/{S3D_NAME}_stage3_predictions.npz")
        progress[S3D_NAME]["stage3_acc"] = S3D_s3_acc
        progress[S3D_NAME]["done"] = True
        save_progress(progress)

        log(f"[{S3D_NAME}] TAMAMLANDI: Stage1={S3D_s1_acc:.4f} "
            f"Stage2={S3D_s2_acc:.4f} Stage3-E2E={S3D_s3_acc:.4f}")

        del S3D_gated
        gpu_cleanup()
        S3D_done = True

    except torch.cuda.OutOfMemoryError:
        gpu_cleanup()
        S3D_batch_size = max(S3D_batch_size // 2, 4)
        log(f"[{S3D_NAME}] !! OOM -- batch_size={S3D_batch_size} ile tekrar deneniyor")
        S3D_train_loader, S3D_train_eval_loader, S3D_val_loader = make_loaders(S3D_batch_size)

    except Exception as e:
        log(f"[{S3D_NAME}] HATA: {repr(e)}")
        progress[S3D_NAME]["error"] = str(e)
        save_progress(progress)
        raise

# ---- Oracle analysis (for this architecture) ----
S3D_oracle_summary = oracle_analysis(S3D_NAME)

print(f"\n✓ {S3D_NAME} tamamlandı ve oracle analizi kaydedildi -> {ORACLE_DIR}")

In [ ]:
# ============================================================
# CELL 12/20 — mvit_v2_s DATA LOADER (MViT-V2-S -- Transformer family)
# Transformer backbones destabilize at large LR -- backbone LR halved (5e-5) and warmup extended (0.10), standard practice for ViT training. batch=16 because of the 224px input plus attention's memory cost.
# Run after Cell 3 (RAM cache + shared definitions).
# ============================================================

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("12 - mvit_v2_s VERİ YÜKLEYİCİ")

MVIT_V2_S_NAME = "mvit_v2_s"
MVIT_V2_S_CTOR = video_models.mvit_v2_s
MVIT_V2_S_WEIGHTS = video_models.MViT_V2_S_Weights.KINETICS400_V1
MVIT_V2_S_RESIZE_TO = 224      # native Kinetics resolution
MVIT_V2_S_BATCH_SIZE = 16
MVIT_V2_S_BACKBONE_LR = 5e-5
MVIT_V2_S_GATE_LR = 2e-3
MVIT_V2_S_WARMUP_FRAC = 0.10

MVIT_V2_S_train_loader, MVIT_V2_S_train_eval_loader, MVIT_V2_S_val_loader = make_loaders(MVIT_V2_S_BATCH_SIZE)

print(f"✓ {MVIT_V2_S_NAME} data loader hazır: resize_to={MVIT_V2_S_RESIZE_TO} "
      f"batch_size={MVIT_V2_S_BATCH_SIZE} backbone_lr={MVIT_V2_S_BACKBONE_LR} "
      f"warmup_frac={MVIT_V2_S_WARMUP_FRAC}")
print(f"  train_batches={len(MVIT_V2_S_train_loader)} val_batches={len(MVIT_V2_S_val_loader)}")

In [ ]:
# ============================================================
# CELL 13/20 — mvit_v2_s: 3-STAGE TRAINING + ORACLE ANALYSIS
# Run after Cell 12 (its own loader must be ready).
# Stage1(25) -> Stage2(25, gate-only) -> Stage3(25, end-to-end)
# ============================================================

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("13 - mvit_v2_s EĞİTİMİ")

progress.setdefault(MVIT_V2_S_NAME, {})
MVIT_V2_S_batch_size = MVIT_V2_S_BATCH_SIZE
MVIT_V2_S_done = False

while not MVIT_V2_S_done:
    try:
        gpu_cleanup()

        # ---- STAGE 1: independent, parallel, one loss + one optimizer each ----
        MVIT_V2_S_model_ir, MVIT_V2_S_model_depth, MVIT_V2_S_s1_acc = train_stage1(
            MVIT_V2_S_NAME, MVIT_V2_S_CTOR, MVIT_V2_S_WEIGHTS, MVIT_V2_S_RESIZE_TO,
            MVIT_V2_S_train_loader, MVIT_V2_S_train_eval_loader, MVIT_V2_S_val_loader,
            backbone_lr=MVIT_V2_S_BACKBONE_LR, warmup_frac=MVIT_V2_S_WARMUP_FRAC,
        )
        progress[MVIT_V2_S_NAME]["stage1_acc"] = MVIT_V2_S_s1_acc
        save_progress(progress)
        del MVIT_V2_S_model_ir, MVIT_V2_S_model_depth
        gpu_cleanup()

        # ---- GATED MODEL, initialized from Stage1 weights ----
        MVIT_V2_S_gated = GatedModel(MVIT_V2_S_CTOR, MVIT_V2_S_WEIGHTS, MVIT_V2_S_RESIZE_TO, N_CLASSES).to(DEVICE)
        MVIT_V2_S_ck_ir = torch.load(f"{CKPT_DIR}/{MVIT_V2_S_NAME}_stage1_ir.pt", map_location="cpu")
        MVIT_V2_S_ck_depth = torch.load(f"{CKPT_DIR}/{MVIT_V2_S_NAME}_stage1_depth.pt", map_location="cpu")
        MVIT_V2_S_gated.backbone_ir.load_state_dict(MVIT_V2_S_ck_ir["backbone"])
        MVIT_V2_S_gated.backbone_depth.load_state_dict(MVIT_V2_S_ck_depth["backbone"])
        MVIT_V2_S_gated.head_ir.load_state_dict(MVIT_V2_S_ck_ir["head"])
        MVIT_V2_S_gated.head_depth.load_state_dict(MVIT_V2_S_ck_depth["head"])

        # ---- STAGE 2: gate-only, backbone frozen ----
        MVIT_V2_S_gated, MVIT_V2_S_s2_acc = train_gated_stage(
            MVIT_V2_S_gated, MVIT_V2_S_train_loader, MVIT_V2_S_train_eval_loader, MVIT_V2_S_val_loader,
            "stage2", MVIT_V2_S_NAME, freeze_backbone=True,
            backbone_lr=MVIT_V2_S_BACKBONE_LR, gate_lr=MVIT_V2_S_GATE_LR, warmup_frac=MVIT_V2_S_WARMUP_FRAC,
        )
        progress[MVIT_V2_S_NAME]["stage2_acc"] = MVIT_V2_S_s2_acc
        save_progress(progress)
        gpu_cleanup()   # GPU cleanup between Stage2 and Stage3

        # ---- STAGE 3: end-to-end ----
        MVIT_V2_S_gated, MVIT_V2_S_s3_acc = train_gated_stage(
            MVIT_V2_S_gated, MVIT_V2_S_train_loader, MVIT_V2_S_train_eval_loader, MVIT_V2_S_val_loader,
            "stage3", MVIT_V2_S_NAME, freeze_backbone=False,
            backbone_lr=MVIT_V2_S_BACKBONE_LR, gate_lr=MVIT_V2_S_GATE_LR, warmup_frac=MVIT_V2_S_WARMUP_FRAC,
        )
        dump_predictions_gated(MVIT_V2_S_gated, MVIT_V2_S_val_loader, f"{PRED_DIR}/{MVIT_V2_S_NAME}_stage3_predictions.npz")
        progress[MVIT_V2_S_NAME]["stage3_acc"] = MVIT_V2_S_s3_acc
        progress[MVIT_V2_S_NAME]["done"] = True
        save_progress(progress)

        log(f"[{MVIT_V2_S_NAME}] TAMAMLANDI: Stage1={MVIT_V2_S_s1_acc:.4f} "
            f"Stage2={MVIT_V2_S_s2_acc:.4f} Stage3-E2E={MVIT_V2_S_s3_acc:.4f}")

        del MVIT_V2_S_gated
        gpu_cleanup()
        MVIT_V2_S_done = True

    except torch.cuda.OutOfMemoryError:
        gpu_cleanup()
        MVIT_V2_S_batch_size = max(MVIT_V2_S_batch_size // 2, 4)
        log(f"[{MVIT_V2_S_NAME}] !! OOM -- batch_size={MVIT_V2_S_batch_size} ile tekrar deneniyor")
        MVIT_V2_S_train_loader, MVIT_V2_S_train_eval_loader, MVIT_V2_S_val_loader = make_loaders(MVIT_V2_S_batch_size)

    except Exception as e:
        log(f"[{MVIT_V2_S_NAME}] HATA: {repr(e)}")
        progress[MVIT_V2_S_NAME]["error"] = str(e)
        save_progress(progress)
        raise

# ---- Oracle analysis (for this architecture) ----
MVIT_V2_S_oracle_summary = oracle_analysis(MVIT_V2_S_NAME)

print(f"\n✓ {MVIT_V2_S_NAME} tamamlandı ve oracle analizi kaydedildi -> {ORACLE_DIR}")

In [ ]:
# ============================================================
# CELL 14/20 — swin3d_t DATA LOADER (Swin3D-T -- Transformer family)
# Transformer backbone -- same reasoning as MViT: low LR (5e-5) + long warmup (0.10). batch=24 because of the 224px cost.
# Run after Cell 3 (RAM cache + shared definitions).
# ============================================================

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("14 - swin3d_t VERİ YÜKLEYİCİ")

SWIN3D_T_NAME = "swin3d_t"
SWIN3D_T_CTOR = video_models.swin3d_t
SWIN3D_T_WEIGHTS = video_models.Swin3D_T_Weights.KINETICS400_V1
SWIN3D_T_RESIZE_TO = 224      # native Kinetics resolution
SWIN3D_T_BATCH_SIZE = 24
SWIN3D_T_BACKBONE_LR = 5e-5
SWIN3D_T_GATE_LR = 2e-3
SWIN3D_T_WARMUP_FRAC = 0.10

SWIN3D_T_train_loader, SWIN3D_T_train_eval_loader, SWIN3D_T_val_loader = make_loaders(SWIN3D_T_BATCH_SIZE)

print(f"✓ {SWIN3D_T_NAME} data loader hazır: resize_to={SWIN3D_T_RESIZE_TO} "
      f"batch_size={SWIN3D_T_BATCH_SIZE} backbone_lr={SWIN3D_T_BACKBONE_LR} "
      f"warmup_frac={SWIN3D_T_WARMUP_FRAC}")
print(f"  train_batches={len(SWIN3D_T_train_loader)} val_batches={len(SWIN3D_T_val_loader)}")

In [ ]:
# ============================================================
# CELL 15/20 — swin3d_t: 3-STAGE TRAINING + ORACLE ANALYSIS
# Run after Cell 14 (its own loader must be ready).
# Stage1(25) -> Stage2(25, gate-only) -> Stage3(25, end-to-end)
# ============================================================

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("15 - swin3d_t EĞİTİMİ")

progress.setdefault(SWIN3D_T_NAME, {})
SWIN3D_T_batch_size = SWIN3D_T_BATCH_SIZE
SWIN3D_T_done = False

while not SWIN3D_T_done:
    try:
        gpu_cleanup()

        # ---- STAGE 1: independent, parallel, one loss + one optimizer each ----
        SWIN3D_T_model_ir, SWIN3D_T_model_depth, SWIN3D_T_s1_acc = train_stage1(
            SWIN3D_T_NAME, SWIN3D_T_CTOR, SWIN3D_T_WEIGHTS, SWIN3D_T_RESIZE_TO,
            SWIN3D_T_train_loader, SWIN3D_T_train_eval_loader, SWIN3D_T_val_loader,
            backbone_lr=SWIN3D_T_BACKBONE_LR, warmup_frac=SWIN3D_T_WARMUP_FRAC,
        )
        progress[SWIN3D_T_NAME]["stage1_acc"] = SWIN3D_T_s1_acc
        save_progress(progress)
        del SWIN3D_T_model_ir, SWIN3D_T_model_depth
        gpu_cleanup()

        # ---- GATED MODEL, initialized from Stage1 weights ----
        SWIN3D_T_gated = GatedModel(SWIN3D_T_CTOR, SWIN3D_T_WEIGHTS, SWIN3D_T_RESIZE_TO, N_CLASSES).to(DEVICE)
        SWIN3D_T_ck_ir = torch.load(f"{CKPT_DIR}/{SWIN3D_T_NAME}_stage1_ir.pt", map_location="cpu")
        SWIN3D_T_ck_depth = torch.load(f"{CKPT_DIR}/{SWIN3D_T_NAME}_stage1_depth.pt", map_location="cpu")
        SWIN3D_T_gated.backbone_ir.load_state_dict(SWIN3D_T_ck_ir["backbone"])
        SWIN3D_T_gated.backbone_depth.load_state_dict(SWIN3D_T_ck_depth["backbone"])
        SWIN3D_T_gated.head_ir.load_state_dict(SWIN3D_T_ck_ir["head"])
        SWIN3D_T_gated.head_depth.load_state_dict(SWIN3D_T_ck_depth["head"])

        # ---- STAGE 2: gate-only, backbone frozen ----
        SWIN3D_T_gated, SWIN3D_T_s2_acc = train_gated_stage(
            SWIN3D_T_gated, SWIN3D_T_train_loader, SWIN3D_T_train_eval_loader, SWIN3D_T_val_loader,
            "stage2", SWIN3D_T_NAME, freeze_backbone=True,
            backbone_lr=SWIN3D_T_BACKBONE_LR, gate_lr=SWIN3D_T_GATE_LR, warmup_frac=SWIN3D_T_WARMUP_FRAC,
        )
        progress[SWIN3D_T_NAME]["stage2_acc"] = SWIN3D_T_s2_acc
        save_progress(progress)
        gpu_cleanup()   # GPU cleanup between Stage2 and Stage3

        # ---- STAGE 3: end-to-end ----
        SWIN3D_T_gated, SWIN3D_T_s3_acc = train_gated_stage(
            SWIN3D_T_gated, SWIN3D_T_train_loader, SWIN3D_T_train_eval_loader, SWIN3D_T_val_loader,
            "stage3", SWIN3D_T_NAME, freeze_backbone=False,
            backbone_lr=SWIN3D_T_BACKBONE_LR, gate_lr=SWIN3D_T_GATE_LR, warmup_frac=SWIN3D_T_WARMUP_FRAC,
        )
        dump_predictions_gated(SWIN3D_T_gated, SWIN3D_T_val_loader, f"{PRED_DIR}/{SWIN3D_T_NAME}_stage3_predictions.npz")
        progress[SWIN3D_T_NAME]["stage3_acc"] = SWIN3D_T_s3_acc
        progress[SWIN3D_T_NAME]["done"] = True
        save_progress(progress)

        log(f"[{SWIN3D_T_NAME}] TAMAMLANDI: Stage1={SWIN3D_T_s1_acc:.4f} "
            f"Stage2={SWIN3D_T_s2_acc:.4f} Stage3-E2E={SWIN3D_T_s3_acc:.4f}")

        del SWIN3D_T_gated
        gpu_cleanup()
        SWIN3D_T_done = True

    except torch.cuda.OutOfMemoryError:
        gpu_cleanup()
        SWIN3D_T_batch_size = max(SWIN3D_T_batch_size // 2, 4)
        log(f"[{SWIN3D_T_NAME}] !! OOM -- batch_size={SWIN3D_T_batch_size} ile tekrar deneniyor")
        SWIN3D_T_train_loader, SWIN3D_T_train_eval_loader, SWIN3D_T_val_loader = make_loaders(SWIN3D_T_batch_size)

    except Exception as e:
        log(f"[{SWIN3D_T_NAME}] HATA: {repr(e)}")
        progress[SWIN3D_T_NAME]["error"] = str(e)
        save_progress(progress)
        raise

# ---- Oracle analysis (for this architecture) ----
SWIN3D_T_oracle_summary = oracle_analysis(SWIN3D_T_NAME)

print(f"\n✓ {SWIN3D_T_NAME} tamamlandı ve oracle analizi kaydedildi -> {ORACLE_DIR}")

In [ ]:
# ============================================================
# CELL 16 (FULL DETAILED ANALYSIS) — 6 ARCHITECTURES x 3 STAGES (Stage1/2/3)
# x (IR+Depth together) ORACLE + REALISTIC FUSION + PER-CLASS +
# PER-SUBJECT ACCURACY ANALYSIS
#
# PREREQUISITE: Cells 0->1->2->3 must have already run in this runtime
# (RAM cache populated: bank_ir, bank_depth, items, train_ids, val_ids ready;
# GatedModel/SingleModel/dump_predictions_*/CKPT_DIR/PRED_DIR/ORACLE_DIR
# ready). You do NOT need to run any per-architecture loader/train cell
# (4-15) -- this cell contains its own CTOR/WEIGHTS/RESIZE_TO definitions.
#
# WHAT IT DOES:
#  - Uses Stage1 and Stage3 predictions AS-IS if they're already on Drive
#    (doesn't recompute them).
#  - Stage2's full softmax was NEVER saved (only the checkpoint + a scalar
#    accuracy existed) -- so it loads the Stage2 checkpoint and runs a
#    ONE-TIME inference pass over the val set (NO training).
#  - For each architecture x each stage: IR acc, Depth acc, hard-oracle,
#    realistic fusion (softmax average), PER-CLASS accuracy (40 classes,
#    full to CSV, worst 5 to console), PER-SUBJECT accuracy (val subjects,
#    all of them to console -- there are few enough to fit).
# ============================================================

import torch.nn.functional as F

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("21 - TAM 3-AŞAMA DETAYLI ANALİZ (6 mimari x Stage1/2/3)")

ARCH_CONFIGS = {
    "r2plus1d_18": (video_models.r2plus1d_18, video_models.R2Plus1D_18_Weights.KINETICS400_V1, 112),
    "r3d_18":      (video_models.r3d_18,      video_models.R3D_18_Weights.KINETICS400_V1,      112),
    "mc3_18":      (video_models.mc3_18,      video_models.MC3_18_Weights.KINETICS400_V1,      112),
    "s3d":         (video_models.s3d,         video_models.S3D_Weights.KINETICS400_V1,          224),
    "mvit_v2_s":   (video_models.mvit_v2_s,   video_models.MViT_V2_S_Weights.KINETICS400_V1,    224),
    "swin3d_t":    (video_models.swin3d_t,    video_models.Swin3D_T_Weights.KINETICS400_V1,     224),
}

# ---- a SINGLE shared val_loader for all architectures -- resizing happens
# inside each model's own forward pass (BackboneWrapper), nothing at the
# dataset/loader level is architecture-specific, so one loader is enough
# for inference.
INFER_BATCH_SIZE = 16
_, _, shared_val_loader = make_loaders(INFER_BATCH_SIZE)
log(f"✓ Ortak val_loader hazır (batch={INFER_BATCH_SIZE}, {len(shared_val_loader)} batch)")

def ensure_stage1(name, ctor, weights, resize_to):
    path = f"{PRED_DIR}/{name}_stage1_predictions.npz"
    if os.path.exists(path):
        return path
    ir_ck, depth_ck = f"{CKPT_DIR}/{name}_stage1_ir.pt", f"{CKPT_DIR}/{name}_stage1_depth.pt"
    if not (os.path.exists(ir_ck) and os.path.exists(depth_ck)):
        return None
    model_ir = SingleModel(ctor, weights, resize_to, N_CLASSES).to(DEVICE)
    model_depth = SingleModel(ctor, weights, resize_to, N_CLASSES).to(DEVICE)
    ck_ir, ck_depth = torch.load(ir_ck, map_location=DEVICE), torch.load(depth_ck, map_location=DEVICE)
    model_ir.backbone.load_state_dict(ck_ir["backbone"]); model_ir.head.load_state_dict(ck_ir["head"])
    model_depth.backbone.load_state_dict(ck_depth["backbone"]); model_depth.head.load_state_dict(ck_depth["head"])
    dump_predictions_stage1(model_ir, model_depth, shared_val_loader, path)
    del model_ir, model_depth; gpu_cleanup()
    return path

def ensure_gated_stage(name, ctor, weights, resize_to, stage_label):
    path = f"{PRED_DIR}/{name}_{stage_label}_predictions.npz"
    if os.path.exists(path):
        return path
    ck_path = f"{CKPT_DIR}/{name}_{stage_label}_full.pt"
    if not os.path.exists(ck_path):
        return None
    model = GatedModel(ctor, weights, resize_to, N_CLASSES).to(DEVICE)
    state = torch.load(ck_path, map_location=DEVICE)
    model.load_state_dict(state)
    dump_predictions_gated(model, shared_val_loader, path)
    del model; gpu_cleanup()
    return path

# ---- 1) backfill any missing prediction files (Stage2 was missing for all of them) ----
available = {}
for name, (ctor, weights, resize_to) in ARCH_CONFIGS.items():
    log(f"--- {name}: eksik tahmin dosyaları tamamlanıyor ---")
    p1 = ensure_stage1(name, ctor, weights, resize_to)
    p2 = ensure_gated_stage(name, ctor, weights, resize_to, "stage2")
    p3 = ensure_gated_stage(name, ctor, weights, resize_to, "stage3")
    available[name] = {"stage1": p1, "stage2": p2, "stage3": p3}
    log(f"    stage1={'OK' if p1 else 'YOK'}  stage2={'OK' if p2 else 'YOK'}  stage3={'OK' if p3 else 'YOK'}")

# ---- 2) detailed analysis for each architecture x each stage ----
def analyze_stage(name, stage_label, path):
    if path is None:
        return None
    d = np.load(path)
    order = np.argsort(d["index"])
    idx, label = d["index"][order], d["label"][order]
    probs_ir, probs_depth = d["probs_ir"][order], d["probs_depth"][order]
    pred_ir, pred_depth = probs_ir.argmax(1), probs_depth.argmax(1)
    pred_fused = ((probs_ir + probs_depth) / 2.0).argmax(1)

    ir_correct, depth_correct = pred_ir == label, pred_depth == label
    fused_correct = pred_fused == label
    oracle_correct = ir_correct | depth_correct
    users = np.array([items[int(i)]["user"] for i in idx])

    class_rows = []
    for c in sorted(set(label.tolist())):
        m = label == c
        class_rows.append({"class": int(c), "n": int(m.sum()),
                            "ir_acc": float(ir_correct[m].mean()), "depth_acc": float(depth_correct[m].mean()),
                            "fusion_acc": float(fused_correct[m].mean()), "oracle_acc": float(oracle_correct[m].mean())})
    class_df = pd.DataFrame(class_rows).sort_values("fusion_acc")

    subj_rows = []
    for u in sorted(set(users.tolist())):
        m = users == u
        subj_rows.append({"user": int(u), "n": int(m.sum()),
                           "ir_acc": float(ir_correct[m].mean()), "depth_acc": float(depth_correct[m].mean()),
                           "fusion_acc": float(fused_correct[m].mean()), "oracle_acc": float(oracle_correct[m].mean())})
    subj_df = pd.DataFrame(subj_rows).sort_values("fusion_acc")

    class_df.to_csv(f"{ORACLE_DIR}/{name}_{stage_label}_per_class.csv", index=False)
    subj_df.to_csv(f"{ORACLE_DIR}/{name}_{stage_label}_per_subject.csv", index=False)

    overall = {"ir_acc": float(ir_correct.mean()), "depth_acc": float(depth_correct.mean()),
               "fusion_acc": float(fused_correct.mean()), "oracle_acc": float(oracle_correct.mean())}
    print(f"\n--- {name} / {stage_label}  (N={len(label)}) ---")
    print(f"  Genel: IR={overall['ir_acc']:.4f}  Depth={overall['depth_acc']:.4f}  "
          f"Füzyon={overall['fusion_acc']:.4f}  Oracle={overall['oracle_acc']:.4f}")
    print(f"  En kötü 5 sınıf (füzyon doğruluğuna göre):")
    print("   " + class_df.head(5).to_string(index=False).replace("\n", "\n   "))
    print(f"  Kişi (subject) bazlı doğruluk:")
    print("   " + subj_df.to_string(index=False).replace("\n", "\n   "))
    return {"overall": overall, "class_df": class_df, "subj_df": subj_df}

results = {}
for name in ARCH_CONFIGS:
    results[name] = {}
    for stage_label in ["stage1", "stage2", "stage3"]:
        results[name][stage_label] = analyze_stage(name, stage_label, available[name][stage_label])

# ---- 3) summary table across all architectures x stages ----
stage("ÖZET -- TÜM MİMARİLER x 3 AŞAMA")
summary_rows = []
for name in ARCH_CONFIGS:
    for stage_label in ["stage1", "stage2", "stage3"]:
        r = results[name][stage_label]
        if r is None:
            summary_rows.append({"model": name, "stage": stage_label, "ir_acc": None, "depth_acc": None,
                                  "fusion_acc": None, "oracle_acc": None})
            continue
        summary_rows.append({"model": name, "stage": stage_label, **r["overall"]})
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(f"{ORACLE_DIR}/full_3stage_summary.csv", index=False)
print(f"\n✓ Tüm CSV'ler (özet + her mimari/aşama için per-class, per-subject) -> {ORACLE_DIR}")


In [ ]:
# ============================================================
# CELL 17 (FUSION METHODS COMPARISON) — 6 ARCHITECTURES x 3 STAGES,
# MULTIPLE FUSION STRATEGIES
#
# Needs no training/RAM cache/model loading -- only reads the stage1/2/3
# prediction .npz files on Drive (Cell 16 produced all of them).
# `items` (needed for subject info) must still be in RAM -- run this
# after Cell 16, in the same runtime.
#
# UNTRAINED (no learned parameters, fixed/dynamic rule) methods:
#   - avg               : softmax average (what we used up to now)
#   - product_of_experts: multiply the probabilities and normalize (geometric mean)
#   - confidence_weighted: for each sample, give more weight to whichever
#                          modality is more "confident" (higher max softmax) there
#   - max_confidence_select: for each sample, use only the more confident
#                            modality's prediction (hard selection, not soft)
#
# TRAINED (LEARNED on val) methods:
#   - learned_alpha_loso: a fixed mixing weight (alpha) between IR and Depth
#     is SEARCHED on the val set -- but to avoid DATA LEAKAGE, via
#     Leave-One-Subject-Out (LOSO) cross-validation: alpha is found on 3
#     val subjects and tested on the 4th (unseen) subject, averaged over
#     4 folds. So it's "trained" but doesn't overfit to the val set itself.
#     Learns a SINGLE scalar parameter (the simplest "trained" fusion).
#   - learned_logreg_loso: a STRONGER/MORE FLEXIBLE combiner -- the backbone
#     models (r2plus1d_18 etc.) are NOT trained at all here, only a small
#     LogisticRegression classifier is trained on IR+Depth's 40+40=80
#     dimensional softmax output vector (can learn a different weight per
#     class -- much more flexible than a single scalar alpha). Also via
#     LOSO: trained on 3 subjects, tested on the 4th (unseen), averaged
#     over 4 folds -- no data leakage. This is the answer to "train the
#     fusion system without training the models".
# ============================================================

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("22 - FÜZYON YÖNTEMLERİ KARŞILAŞTIRMASI -- 6 MİMARİ x 3 AŞAMA")

ALL_MODEL_NAMES = ["r2plus1d_18", "r3d_18", "mc3_18", "s3d", "mvit_v2_s", "swin3d_t"]
STAGE_LABELS = ["stage1", "stage2", "stage3"]

def fusion_avg(p_ir, p_depth):
    return (p_ir + p_depth) / 2.0

def fusion_product_of_experts(p_ir, p_depth):
    prod = p_ir * p_depth
    prod = prod / prod.sum(axis=1, keepdims=True)
    return prod

def fusion_confidence_weighted(p_ir, p_depth):
    w_ir = p_ir.max(axis=1)
    w_depth = p_depth.max(axis=1)
    wa = (w_ir / (w_ir + w_depth + 1e-9))[:, None]
    return wa * p_ir + (1 - wa) * p_depth

def fusion_max_confidence_select(p_ir, p_depth):
    """Not a soft blend -- a hard selection: for each sample, only the
    prediction of whichever modality is more confident (max softmax) is used."""
    use_ir = p_ir.max(axis=1) >= p_depth.max(axis=1)
    pred = np.where(use_ir, p_ir.argmax(1), p_depth.argmax(1))
    return pred   # returns pred directly, not probabilities

def learned_alpha_loso(p_ir, p_depth, label, users, alphas=np.linspace(0, 1, 21)):
    """LOSO: each val subject is held out in turn, alpha is grid-searched on
    the other subjects, then tested on the held-out subject. No data
    leakage -- alpha is never chosen using the data of the subject it's
    tested on."""
    unique_users = sorted(set(users.tolist()))
    fold_correct, fold_total, chosen_alphas = 0, 0, []
    for held_out in unique_users:
        train_mask = users != held_out
        test_mask = users == held_out
        best_alpha, best_acc = 0.5, -1
        for a in alphas:
            mix = a * p_ir[train_mask] + (1 - a) * p_depth[train_mask]
            acc = (mix.argmax(1) == label[train_mask]).mean()
            if acc > best_acc:
                best_acc, best_alpha = acc, a
        mix_test = best_alpha * p_ir[test_mask] + (1 - best_alpha) * p_depth[test_mask]
        fold_correct += (mix_test.argmax(1) == label[test_mask]).sum()
        fold_total += test_mask.sum()
        chosen_alphas.append(best_alpha)
    return fold_correct / fold_total, chosen_alphas

from sklearn.linear_model import LogisticRegression

def learned_logreg_loso(p_ir, p_depth, label, users):
    """A MORE FLEXIBLE trained fusion: a LogisticRegression is trained on
    [p_ir || p_depth] (80-dimensional) -- it can learn a separate weight per
    class (unlike a single scalar alpha). Data leakage is prevented via
    LOSO: trained on 3 subjects, tested on the 4th (unseen), averaged over 4
    folds. The backbone models (r2plus1d_18 etc.) are NOT trained at all
    here -- only this small combiner layer is trained, on fixed softmax
    outputs."""
    X = np.concatenate([p_ir, p_depth], axis=1)  # (N, 2*n_classes)
    unique_users = sorted(set(users.tolist()))
    fold_correct, fold_total = 0, 0
    for held_out in unique_users:
        train_mask = users != held_out
        test_mask = users == held_out
        clf = LogisticRegression(max_iter=2000, C=1.0, multi_class="auto")
        clf.fit(X[train_mask], label[train_mask])
        pred_test = clf.predict(X[test_mask])
        fold_correct += (pred_test == label[test_mask]).sum()
        fold_total += test_mask.sum()
    return fold_correct / fold_total

rows = []
for name in ALL_MODEL_NAMES:
    for stage_label in STAGE_LABELS:
        path = f"{PRED_DIR}/{name}_{stage_label}_predictions.npz"
        if not os.path.exists(path):
            continue
        d = np.load(path)
        order = np.argsort(d["index"])
        idx, label = d["index"][order], d["label"][order]
        p_ir, p_depth = d["probs_ir"][order], d["probs_depth"][order]
        users = np.array([items[int(i)]["user"] for i in idx])

        acc_avg = (fusion_avg(p_ir, p_depth).argmax(1) == label).mean()
        acc_poe = (fusion_product_of_experts(p_ir, p_depth).argmax(1) == label).mean()
        acc_conf = (fusion_confidence_weighted(p_ir, p_depth).argmax(1) == label).mean()
        acc_maxsel = (fusion_max_confidence_select(p_ir, p_depth) == label).mean()
        acc_loso, chosen_alphas = learned_alpha_loso(p_ir, p_depth, label, users)
        acc_logreg = learned_logreg_loso(p_ir, p_depth, label, users)

        methods = {"avg": acc_avg, "product_of_experts": acc_poe,
                   "confidence_weighted": acc_conf, "max_confidence_select": acc_maxsel,
                   "learned_alpha_loso": acc_loso, "learned_logreg_loso": acc_logreg}
        best_method = max(methods, key=methods.get)

        rows.append({"model": name, "stage": stage_label,
                      "avg": acc_avg, "product_of_experts": acc_poe,
                      "confidence_weighted": acc_conf, "max_confidence_select": acc_maxsel,
                      "learned_alpha_loso": acc_loso,
                      "loso_mean_alpha": float(np.mean(chosen_alphas)),
                      "learned_logreg_loso": acc_logreg,
                      "best_method": best_method, "best_acc": methods[best_method]})

fusion_methods_df = pd.DataFrame(rows)
print(fusion_methods_df.to_string(index=False))

out_path = f"{ORACLE_DIR}/fusion_methods_comparison.csv"
fusion_methods_df.to_csv(out_path, index=False)
print(f"\n✓ Kaydedildi -> {out_path}")

stage("ÖZET")
print("Her mimari x aşama için en iyi füzyon yöntemi:")
print(fusion_methods_df[["model", "stage", "best_method", "best_acc"]].to_string(index=False))

# -- is "avg" (what we've used so far) always the best? --
n_avg_best = int((fusion_methods_df["best_method"] == "avg").sum())
print(f"\n>>> {n_avg_best}/{len(fusion_methods_df)} kombinasyonda basit ortalama zaten en iyi yöntemdi.")
print(f">>> Diğer {len(fusion_methods_df) - n_avg_best} kombinasyonda alternatif bir füzyon yöntemi daha iyi çıktı "
      f"-- bunlara özellikle bak, raporda 'avg yerine bunu kullandık' diyebiliriz.")

# -- does the more flexible trained method (logreg) beat the single-scalar trained method (alpha), and how often? --
n_logreg_beats_alpha = int((fusion_methods_df["learned_logreg_loso"] > fusion_methods_df["learned_alpha_loso"]).sum())
n_logreg_best = int((fusion_methods_df["best_method"] == "learned_logreg_loso").sum())
print(f"\n>>> learned_logreg_loso (esnek, sınıf-bazlı ağırlıklı), {n_logreg_beats_alpha}/{len(fusion_methods_df)} "
      f"kombinasyonda learned_alpha_loso'yu (tek skaler ağırlık) geçti.")
print(f">>> learned_logreg_loso, {n_logreg_best}/{len(fusion_methods_df)} kombinasyonda TÜM yöntemler içinde en iyisiydi.")
print(fusion_methods_df[["model", "stage", "learned_alpha_loso", "learned_logreg_loso"]].to_string(index=False))


In [ ]:
# ============================================================
# CELL 18 — FINAL PER-MODEL FUSION SYSTEM (SINGLE CELL)
#
# The earlier logreg (on the raw 80-dim softmax) FLOPPED -- overfit.
# Reason: small train set per LOSO fold (~3 subjects) + 80 dims is too
# many parameters. Fix: instead of raw softmax, extract a small per-sample
# "behavior summary" (7 dims) -- how confident IR is, how confident Depth
# is, whether they agree, etc. -- and learn on top of that. This is a real
# system that "learns when each model tends to be wrong"; with 7 dims
# instead of 80 it learns without memorizing even on small data.
#
# 6 methods are compared (5 fixed rules + 1 learned gate):
#   avg / product_of_experts / confidence_weighted / max_confidence_select
#   learned_alpha        : a single scalar mixing weight (LOSO)
#   learned_meta_gate     : PER-SAMPLE weight -- a small LogisticRegression,
#     input = [IR_confidence, Depth_confidence, IR_margin, Depth_margin,
#     IR_entropy, Depth_entropy, do_they_agree], target = "is IR or Depth
#     right on this sample" (via LOSO, no data leakage). So the model
#     learns each model's characteristic error pattern (e.g. "IR is
#     confident but wrong in this kind of scene").
#
# The best method is picked per (architecture, stage) and written to a
# lookup + made deployable via `apply_best_fusion(name, stage_label, p_ir, p_depth)`.
#
# PREREQUISITE: Cell 16 must have run in this runtime.
# ============================================================

import json
from sklearn.linear_model import LogisticRegression

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

stage("23 - MODELE ÖZEL NİHAİ FÜZYON SİSTEMİ (+ ÖĞRENİLMİŞ DAVRANIŞ GATE'İ)")

ALL_MODEL_NAMES = ["r2plus1d_18", "r3d_18", "mc3_18", "s3d", "mvit_v2_s", "swin3d_t"]
STAGE_LABELS = ["stage1", "stage2", "stage3"]

# ---- 4 fixed-rule methods ----
def fusion_avg(p_ir, p_depth):
    return (p_ir + p_depth) / 2.0

def fusion_product_of_experts(p_ir, p_depth):
    prod = p_ir * p_depth
    return prod / prod.sum(axis=1, keepdims=True)

def fusion_confidence_weighted(p_ir, p_depth):
    w_ir, w_depth = p_ir.max(axis=1), p_depth.max(axis=1)
    wa = (w_ir / (w_ir + w_depth + 1e-9))[:, None]
    return wa * p_ir + (1 - wa) * p_depth

def fusion_max_confidence_select(p_ir, p_depth):
    use_ir = p_ir.max(axis=1) >= p_depth.max(axis=1)
    return np.where(use_ir, p_ir.argmax(1), p_depth.argmax(1))

# ---- learned_alpha: a single scalar weight, LOSO ----
def best_alpha_loso(p_ir, p_depth, label, users, alphas=np.linspace(0, 1, 21)):
    chosen = []
    for held_out in sorted(set(users.tolist())):
        train_mask = users != held_out
        best_a, best_acc = 0.5, -1
        for a in alphas:
            mix = a * p_ir[train_mask] + (1 - a) * p_depth[train_mask]
            acc = (mix.argmax(1) == label[train_mask]).mean()
            if acc > best_acc:
                best_acc, best_a = acc, a
        chosen.append(best_a)
    return float(np.mean(chosen))

# ---- learned_meta_gate: a per-sample gate learned on a small, low-
# dimensional "behavior" feature. 7 dims instead of raw 80-dim softmax --
# doesn't overfit, and learns each model's OWN characteristic failure modes.
def meta_features(p_ir, p_depth):
    sorted_ir, sorted_depth = np.sort(p_ir, axis=1), np.sort(p_depth, axis=1)
    margin_ir = sorted_ir[:, -1] - sorted_ir[:, -2]
    margin_depth = sorted_depth[:, -1] - sorted_depth[:, -2]
    ent_ir = -(p_ir * np.log(p_ir + 1e-9)).sum(1)
    ent_depth = -(p_depth * np.log(p_depth + 1e-9)).sum(1)
    agree = (p_ir.argmax(1) == p_depth.argmax(1)).astype(float)
    return np.stack([p_ir.max(1), p_depth.max(1), margin_ir, margin_depth,
                      ent_ir, ent_depth, agree], axis=1)

def meta_gate_loso(p_ir, p_depth, label, users):
    """Target: 'is IR or Depth correct on this sample' (if both or neither
    is correct, use whichever is more confident as a weak label). LOSO:
    train on 3 subjects, test on the 4th -> averaged over 4 folds. Also
    retrains the final production model on ALL of the data and returns it."""
    X = meta_features(p_ir, p_depth)
    ir_ok = p_ir.argmax(1) == label
    depth_ok = p_depth.argmax(1) == label
    y = np.where(ir_ok & ~depth_ok, 1, np.where(depth_ok & ~ir_ok, 0,
                 (p_ir.max(1) >= p_depth.max(1)).astype(int)))
    fold_correct, fold_total = 0, 0
    for held_out in sorted(set(users.tolist())):
        train_mask, test_mask = users != held_out, users == held_out
        clf = LogisticRegression(max_iter=1000, C=0.5)
        clf.fit(X[train_mask], y[train_mask])
        w_ir = clf.predict_proba(X[test_mask])[:, 1][:, None]
        mix = w_ir * p_ir[test_mask] + (1 - w_ir) * p_depth[test_mask]
        fold_correct += (mix.argmax(1) == label[test_mask]).sum()
        fold_total += test_mask.sum()
    prod_clf = LogisticRegression(max_iter=1000, C=0.5).fit(X, y)  # trained on all data, for deployment
    return fold_correct / fold_total, prod_clf

# ---- compare 6 methods for each (model, stage) ----
rows, lookup, prod_clfs = [], {}, {}
for name in ALL_MODEL_NAMES:
    for stage_label in STAGE_LABELS:
        path = f"{PRED_DIR}/{name}_{stage_label}_predictions.npz"
        if not os.path.exists(path):
            continue
        d = np.load(path)
        order = np.argsort(d["index"])
        idx, label = d["index"][order], d["label"][order]
        p_ir, p_depth = d["probs_ir"][order], d["probs_depth"][order]
        users = np.array([items[int(i)]["user"] for i in idx])

        alpha = best_alpha_loso(p_ir, p_depth, label, users)
        acc_meta, clf = meta_gate_loso(p_ir, p_depth, label, users)
        methods = {
            "avg": (fusion_avg(p_ir, p_depth).argmax(1) == label).mean(),
            "product_of_experts": (fusion_product_of_experts(p_ir, p_depth).argmax(1) == label).mean(),
            "confidence_weighted": (fusion_confidence_weighted(p_ir, p_depth).argmax(1) == label).mean(),
            "max_confidence_select": (fusion_max_confidence_select(p_ir, p_depth) == label).mean(),
            "learned_alpha": (alpha * p_ir + (1 - alpha) * p_depth).argmax(1).__eq__(label).mean(),
            "learned_meta_gate": acc_meta,
        }
        best_method = max(methods, key=methods.get)
        rows.append({"model": name, "stage": stage_label, **methods,
                      "alpha": alpha, "best_method": best_method, "best_acc": methods[best_method]})
        key = f"{name}__{stage_label}"
        lookup[key] = {"method": best_method, "alpha": alpha, "acc": float(methods[best_method])}
        prod_clfs[key] = clf

final_df = pd.DataFrame(rows)
print(final_df.to_string(index=False))

lookup_path = f"{ORACLE_DIR}/best_fusion_lookup.json"
with open(lookup_path, "w") as f:
    json.dump(lookup, f, indent=2)
final_df.to_csv(f"{ORACLE_DIR}/best_fusion_final.csv", index=False)
print(f"\n✓ Lookup -> {lookup_path}   |   Tablo -> {ORACLE_DIR}/best_fusion_final.csv")

# ---- DEPLOY: automatically applies the right method per model+stage ----
def apply_best_fusion(name, stage_label, p_ir, p_depth):
    """Returns: (pred, method_name). No training/model needed -- just takes
    the softmax vectors."""
    key = f"{name}__{stage_label}"
    method, alpha = lookup[key]["method"], lookup[key]["alpha"]
    if method == "max_confidence_select":
        return fusion_max_confidence_select(p_ir, p_depth), method
    if method == "learned_alpha":
        return (alpha * p_ir + (1 - alpha) * p_depth).argmax(1), method
    if method == "learned_meta_gate":
        w_ir = prod_clfs[key].predict_proba(meta_features(p_ir, p_depth))[:, 1][:, None]
        return (w_ir * p_ir + (1 - w_ir) * p_depth).argmax(1), method
    fn = {"avg": fusion_avg, "product_of_experts": fusion_product_of_experts,
          "confidence_weighted": fusion_confidence_weighted}[method]
    return fn(p_ir, p_depth).argmax(1), method

stage("ÖZET -- HER MİMARİ İÇİN NİHAİ ÜRETİM YÖNTEMİ")
print(final_df[["model", "stage", "best_method", "best_acc"]].to_string(index=False))
n_meta_best = int((final_df["best_method"] == "learned_meta_gate").sum())
print(f"\n>>> Öğrenilmiş davranış-gate'i (learned_meta_gate), {n_meta_best}/{len(final_df)} kombinasyonda en iyisiydi.")
print(">>> Kullanım: pred, method = apply_best_fusion('mvit_v2_s', 'stage3', p_ir, p_depth)")


In [ ]:
# ============================================================
# CELL 19 — COLLECT ALL NEEDED FILES INTO A SINGLE DRIVE FOLDER
# (preparation before uploading to Hugging Face)
#
# Copies the 3 folders currently scattered on Drive (checkpoints,
# predictions, oracle) + progress.json into a SINGLE "export package"
# folder. Doesn't delete/move anything -- the originals stay where they
# are, this only copies. After running this cell, send the EXPORT_ROOT
# link so the Hugging Face upload code can be added to the notebook.
#
# PREREQUISITE: only needs Drive mounted (done in Cell 0) -- no RAM
# cache/model needed, runs independently.
# ============================================================

import shutil, os, time

def stage(text):
    print("\n" + "=" * 75, flush=True)
    print(text, flush=True)
    print("=" * 75, flush=True)

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

stage("24 - TÜM GEREKLİ DOSYALARI TEK KLASÖRE TOPLA")

# DRIVE_ROOT / CKPT_DIR / PRED_DIR / ORACLE_DIR / PROGRESS_PATH come from
# Cell 0 (path config) -- no need to redefine them here.
EXPORT_ROOT = "/content/drive/MyDrive/CUHKX_cache/hf_export_package"
os.makedirs(EXPORT_ROOT, exist_ok=True)

def dir_stats(path):
    n, total = 0, 0
    for r, _, fs in os.walk(path):
        for f in fs:
            n += 1
            total += os.path.getsize(os.path.join(r, f))
    return n, total

def copy_dir(src, dst_name):
    dst = f"{EXPORT_ROOT}/{dst_name}"
    if not os.path.exists(src):
        log(f"!! YOK, atlanıyor: {src}")
        return
    log(f"kopyalanıyor: {src} -> {dst}")
    shutil.copytree(src, dst, dirs_exist_ok=True)
    n, total = dir_stats(dst)
    log(f"✓ {dst_name}: {n} dosya, {total/1e6:.1f} MB")

copy_dir(CKPT_DIR, "checkpoints")
copy_dir(PRED_DIR, "predictions")
copy_dir(ORACLE_DIR, "oracle")

if os.path.exists(PROGRESS_PATH):
    shutil.copy2(PROGRESS_PATH, f"{EXPORT_ROOT}/progress.json")
    log("✓ progress.json kopyalandı")
else:
    log("!! progress.json bulunamadı, atlanıyor")

# ---- total size + top 10 largest files (for planning the HF upload) ----
stage("PAKET ÖZETİ")
all_files = []
for r, _, fs in os.walk(EXPORT_ROOT):
    for f in fs:
        p = os.path.join(r, f)
        all_files.append((os.path.getsize(p), p))
all_files.sort(reverse=True)

total_bytes = sum(s for s, _ in all_files)
print(f"TOPLAM: {len(all_files)} dosya, {total_bytes/1e9:.2f} GB")
print(f"Export klasörü: {EXPORT_ROOT}\n")
print("En büyük 10 dosya:")
for size, path in all_files[:10]:
    print(f"  {size/1e6:8.1f} MB  {path.replace(EXPORT_ROOT + '/', '')}")

print(f"\n>>> Bu klasörün Drive paylaşım linkini şimdi at, HuggingFace yükleme kodunu")
print(f">>> notebook'a ekleyip birlikte düzeltelim.")
